# Olist Payment and Commercial Analysis

## Objective

This analysis evaluates how customers pay for delivered orders and how commercial factors such as merchandise value, freight cost, and price range vary across transactions.

The analysis will:

1. Validate the delivered-order payment scope and payment-table grain.
2. Examine payment-method usage and contribution.
3. Analyze installment behavior and order value.
4. Compare payment methods using commercial and customer-experience measures.
5. Evaluate price bands and freight burden.
6. Identify commercially important or potentially problematic segments.
7. Reconcile the resulting tables with the validated delivered-order scope.
8. Export validated tables for reporting and visualization.

## 1. Load and Validate the Payment Analysis Scope

Payment records may contain multiple rows for the same order when an order uses more than one payment transaction or payment method. Therefore, the payment-table grain must be validated before joining payment information with order-level or item-level data. This prevents order values and item measures from being duplicated.

In [3]:
# Import required libraries and define the processed-data path.
import numpy as np
import pandas as pd
from pathlib import Path

processed_data_path = Path("../data/processed")

print(processed_data_path.resolve())

C:\Users\Hp\DataAnalyst\data\processed


In [4]:
# Load the cleaned payment table and analysis-ready delivered-order-item dataset.
order_payments = pd.read_csv(
    processed_data_path / "olist_order_payments_clean.csv"
)

delivered_order_items = pd.read_parquet(
    processed_data_path / "delivered_order_items_analysis_ready.parquet"
)

# Inspect the loaded datasets.
print("Order-payments shape:", order_payments.shape)
print("Delivered-order-items shape:", delivered_order_items.shape)

print("\nOrder-payments columns:")
print(order_payments.columns.tolist())

order_payments.head()

Order-payments shape: (103886, 10)
Delivered-order-items shape: (110197, 74)

Order-payments columns:
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value', 'flag_payment_sequence_mismatch', 'flag_payment_type_not_defined', 'flag_invalid_payment_installments', 'flag_zero_payment_value', 'flag_payment_quality_exception']


,order_id,payment_sequential,payment_type,payment_installments,payment_value,flag_payment_sequence_mismatch,flag_payment_type_not_defined,flag_invalid_payment_installments,flag_zero_payment_value,flag_payment_quality_exception
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33,False,False,False,False,False
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39,False,False,False,False,False
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71,False,False,False,False,False
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,False,False,False,False,False
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45,False,False,False,False,False


### 1.1 Validate the Payment-Table Grain

The payment table is recorded at payment-transaction grain rather than strictly one row per order. An order may have multiple payment records, so row uniqueness, payment-sequence behavior, missing values, and the number of transactions per order must be examined before aggregation or joining.

In [5]:
# Validate the payment-table grain, completeness, and value ranges.
print("Payment-table shape:", order_payments.shape)

print("Distinct order_id values:", order_payments['order_id'].nunique())

print("Duplicate order_id and payment_sequential combinations:", order_payments[['order_id', 'payment_sequential']].duplicated().sum())

print("\nMissing-value counts:", "\n", order_payments.isna().sum())

print("\nColumn datatypes:", "\n", order_payments.dtypes)


Payment-table shape: (103886, 10)
Distinct order_id values: 99440
Duplicate order_id and payment_sequential combinations: 0

Missing-value counts: 
 order_id                             0
payment_sequential                   0
payment_type                         0
payment_installments                 0
payment_value                        0
flag_payment_sequence_mismatch       0
flag_payment_type_not_defined        0
flag_invalid_payment_installments    0
flag_zero_payment_value              0
flag_payment_quality_exception       0
dtype: int64

Column datatypes: 
 order_id                                 str
payment_sequential                     int64
payment_type                             str
payment_installments                   int64
payment_value                        float64
flag_payment_sequence_mismatch          bool
flag_payment_type_not_defined           bool
flag_invalid_payment_installments       bool
flag_zero_payment_value                 bool
flag_payment_quality_e

In [6]:
print("\nPayment-type value counts:", "\n",order_payments['payment_type'].value_counts(dropna=False))

print("\nMinimum payment_installments:", order_payments['payment_installments'].min())

print("Maximum payment_installments:", order_payments['payment_installments'].max())

print("\nMinimum payment_value:", order_payments['payment_value'].min())

print("Maximum payment_value:", order_payments['payment_value'].max())

payment_rows_per_order = order_payments.groupby('order_id').size()

print("\nNumber of orders with more than one payment row:", payment_rows_per_order.gt(1).sum())

print("Maximum number of payment rows for one order:", payment_rows_per_order.max())


Payment-type value counts: 
 payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

Minimum payment_installments: 0
Maximum payment_installments: 24

Minimum payment_value: 0.0
Maximum payment_value: 13664.08

Number of orders with more than one payment row: 2961
Maximum number of payment rows for one order: 29


#### Payment-Table Validation Summary

The cleaned payment table contains 103,886 payment records associated with 99,440 distinct orders. The combination of `order_id` and `payment_sequential` is unique, confirming that each row represents a distinct payment transaction within an order.

A total of 2,961 orders contain multiple payment rows, and one order contains as many as 29 payment transactions. Therefore, the table cannot be joined directly to item-level data without first controlling its analytical grain.

All payment fields and quality flags are complete. Credit card is the most common payment type, followed by boleto, voucher, and debit card. However, the table also contains three `not_defined` records, installment values ranging from 0 to 24, and payment values ranging from 0 to 13,664.08. The existing quality flags must therefore be examined before constructing the delivered-order payment scope.

### 1.2 Inspect Payment Quality Exceptions

The cleaned payment table includes component-level quality flags for payment-sequence mismatches, undefined payment types, invalid installment values, and zero-value payments. These exceptions are examined before filtering to delivered orders so that unusual records remain transparent and are not silently removed from the analysis.

In [7]:
# Inspect payment-quality flags and records associated with each exception.

# All five payment-quality flags, including the overall exception flag.
payment_quality_flag_columns = [
    'flag_payment_sequence_mismatch',
    'flag_payment_type_not_defined',
    'flag_invalid_payment_installments',
    'flag_zero_payment_value',
    'flag_payment_quality_exception'
    ]

# Component flags used to independently validate the overall flag.
payment_quality_component_columns = [
    'flag_payment_sequence_mismatch',
    'flag_payment_type_not_defined',
    'flag_invalid_payment_installments',
    'flag_zero_payment_value'
    ]

# Identify rows containing at least one component-level exception.
has_component_exception = order_payments[payment_quality_component_columns].any(axis=1)

print("Number of True values for every flag:","\n", order_payments[payment_quality_flag_columns].sum())

print("\nTotal number of exception rows:", has_component_exception.sum())

print("Number of distinct affected orders:", order_payments.loc[has_component_exception, 'order_id'].nunique())

print("\nNot-defined payment records:")
display(order_payments[order_payments['flag_payment_type_not_defined']])

print("Records with invalid installments:")
display(order_payments[order_payments['flag_invalid_payment_installments']])

print("Records with zero payment value:")
display(order_payments[order_payments['flag_zero_payment_value']])

print("Every overall exception has at least one component flag:", has_component_exception[order_payments['flag_payment_quality_exception']].all())

print("Every component-level exception activates the overall flag:", order_payments.loc[has_component_exception,'flag_payment_quality_exception'].all())

print("Component exceptions exactly match the overall exception flag:", has_component_exception.equals(order_payments['flag_payment_quality_exception']))

Number of True values for every flag: 
 flag_payment_sequence_mismatch       82
flag_payment_type_not_defined         3
flag_invalid_payment_installments     2
flag_zero_payment_value               9
flag_payment_quality_exception       91
dtype: int64

Total number of exception rows: 91
Number of distinct affected orders: 88

Not-defined payment records:


,order_id,payment_sequential,payment_type,payment_installments,payment_value,flag_payment_sequence_mismatch,flag_payment_type_not_defined,flag_invalid_payment_installments,flag_zero_payment_value,flag_payment_quality_exception
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0,False,True,False,True,True
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0,False,True,False,True,True
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0,False,True,False,True,True


Records with invalid installments:


,order_id,payment_sequential,payment_type,payment_installments,payment_value,flag_payment_sequence_mismatch,flag_payment_type_not_defined,flag_invalid_payment_installments,flag_zero_payment_value,flag_payment_quality_exception
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69,True,False,True,False,True
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94,True,False,True,False,True


Records with zero payment value:


,order_id,payment_sequential,payment_type,payment_installments,payment_value,flag_payment_sequence_mismatch,flag_payment_type_not_defined,flag_invalid_payment_installments,flag_zero_payment_value,flag_payment_quality_exception
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0,False,False,False,True,True
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0,False,False,False,True,True
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0,False,False,False,True,True
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0,False,True,False,True,True
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0,False,True,False,True,True
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0,False,False,False,True,True
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0,False,False,False,True,True
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0,False,True,False,True,True
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0,False,False,False,True,True


Every overall exception has at least one component flag: True
Every component-level exception activates the overall flag: True
Component exceptions exactly match the overall exception flag: True


#### Payment-Quality Exception Summary

The payment-quality flags are internally consistent. A total of 91 payment records across 88 distinct orders contain at least one identified exception, and the component-level flags exactly reproduce the overall payment-quality exception flag.

Sequence mismatches account for most flagged records. Three `not_defined` payment records also have zero payment value, while two credit-card records have invalid zero-installment values and sequence mismatches. Because some records contain multiple exceptions, the sum of the individual flag counts is greater than the number of affected rows.

These records should not be removed automatically. Their presence and effect within the delivered-order scope must first be evaluated so that legitimate payment value is not unintentionally excluded.

### 1.3 Create and Validate the Delivered-Order Payment Scope

In [8]:
# Create one row per delivered order for payment-scope validation.
delivered_order_scope = delivered_order_items.drop_duplicates(subset=['order_id']).copy()

print("Delivered-order scope shape:", delivered_order_scope.shape)
print("Distinct delivered orders:", delivered_order_scope['order_id'].nunique())
print("Duplicate order_id values:", delivered_order_scope['order_id'].duplicated().sum())

Delivered-order scope shape: (96478, 74)
Distinct delivered orders: 96478
Duplicate order_id values: 0


In [9]:
# Validate payment-record coverage for delivered orders and create the delivered-order payment scope.

print("Number of delivered orders with at least one payment record:", delivered_order_scope['payment_record_count'].ge(1).sum())

print("Number of delivered orders without any payment record:", delivered_order_scope['payment_record_count'].lt(1).sum())

print("\nDelivered orders without payment records:")

display(delivered_order_scope[delivered_order_scope['payment_record_count'].lt(1)])

# Identify payment records outside the delivered-order scope.
outside_delivered_scope_mask = ~order_payments['order_id'].isin(delivered_order_scope['order_id'])

print("Number of payment rows outside the delivered-order scope:", outside_delivered_scope_mask.sum())

print("Number of distinct payment-table orders outside the delivered-order scope:", order_payments.loc[outside_delivered_scope_mask, 'order_id'].nunique())

# Retain all payment rows associated with delivered orders.
delivered_order_payments = order_payments[order_payments['order_id'].isin(delivered_order_scope['order_id'])].copy()

print("\nDelivered-order payment row count:",len(delivered_order_payments))

print("Delivered-order payment distinct-order count:", delivered_order_payments['order_id'].nunique())

Number of delivered orders with at least one payment record: 96477
Number of delivered orders without any payment record: 1

Delivered orders without payment records:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,flag_future_shipping_limit_date,customer_id,order_status,...,delivery_difference_days,delivery_calendar_difference_days,delivery_status,freight_burden_percentage,order_purchase_month,order_purchase_year,order_purchase_quarter,order_purchase_month_number,order_purchase_weekday,order_purchase_weekday_number
82572,bfbd0f9bdef84302105ad712db648a6c,1,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83,False,86dc2ffce2dfff336de2f386a786e574,delivered,...,36.324745,36,late,6.290287,2016-09,2016,3,9,Thursday,3


Number of payment rows outside the delivered-order scope: 3130
Number of distinct payment-table orders outside the delivered-order scope: 2963

Delivered-order payment row count: 100756
Delivered-order payment distinct-order count: 96477


#### Delivered-Order Payment Coverage Summary

The delivered-order scope contains 96,478 distinct orders. Payment records are available for 96,477 of those orders, leaving one delivered order without a corresponding payment record.

The missing-payment order, `bfbd0f9bdef84302105ad712db648a6c`, remains part of the validated delivered-order population but cannot contribute to payment-method, installment, or payment-value metrics. Payment-specific calculations will therefore use only orders with available payment records, while overall delivered-order KPIs will retain the complete 96,478-order scope.

The delivered-order payment table contains 100,756 payment records across 96,477 orders, confirming that some delivered orders contain multiple payment transactions. An additional 3,130 payment rows from 2,963 orders fall outside the delivered-order scope and are excluded from this analysis.

In [10]:
# Validate the delivered-order payment scope and its payment-quality exceptions.
print("Duplicate order_id and payment_sequential combinations:", delivered_order_payments[['order_id', 'payment_sequential']].duplicated().sum())
print("Missing values across the table: ", delivered_order_payments.isna().sum().sum())
print("Number of orders with multiple payment rows: ",(delivered_order_payments.groupby('order_id').agg(payment_count=('payment_value', 'count'))).gt(1).sum())
print("Maximum payment rows for one delivered order: ", (delivered_order_payments.groupby('order_id').agg(payment_count=('payment_value', 'count'))).max())
print("\nPayment-type value counts: ","\n", delivered_order_payments['payment_type'].value_counts())
print("\nPayment-quality exception counts within the delivered scope:", "\n",delivered_order_payments[payment_quality_flag_columns].sum())

delivered_payment_exception_rows = delivered_order_payments[delivered_order_payments['flag_payment_quality_exception']]
print("Distinct delivered orders affected by payment-quality exceptions:", delivered_payment_exception_rows['order_id'].nunique())

Duplicate order_id and payment_sequential combinations: 0
Missing values across the table:  0
Number of orders with multiple payment rows:  payment_count    2875
dtype: int64
Maximum payment rows for one delivered order:  payment_count    26
dtype: int64

Payment-type value counts:  
 payment_type
credit_card    74586
boleto         19191
voucher         5493
debit_card      1486
Name: count, dtype: int64

Payment-quality exception counts within the delivered scope: 
 flag_payment_sequence_mismatch       80
flag_payment_type_not_defined         0
flag_invalid_payment_installments     2
flag_zero_payment_value               4
flag_payment_quality_exception       84
dtype: int64
Distinct delivered orders affected by payment-quality exceptions: 82


#### Delivered-Order Payment-Scope Validation Summary

The delivered-order payment scope contains 100,756 payment records representing 96,477 delivered orders. The combination of `order_id` and `payment_sequential` remains unique, and the table contains no missing values.

A total of 2,875 delivered orders use more than one payment record, with a maximum of 26 payment rows associated with a single delivered order. Credit card is the dominant payment type, followed by boleto, voucher, and debit card.

Within the delivered scope, 84 payment rows across 82 distinct orders contain a payment-quality exception. Most exceptions relate to payment-sequence mismatches. Two records have invalid installment values, four have zero payment value, and none of the delivered-order payment records use the `not_defined` payment type.

The exception records will be retained at this stage because their potential effect on order-level payment totals must be evaluated before any exclusion decision is made.

## 2. Create the Order-Level Payment Analysis Table

Payment transactions must be aggregated to one row per delivered order before they can be combined with order-level commercial, delivery, review, or customer measures. This prevents orders with multiple payment records from receiving disproportionate weight in subsequent analysis.

In [11]:
# Aggregate payment transactions to one row per delivered order and create multi-payment indicators.
order_payment_summary = delivered_order_payments.groupby('order_id').agg(payment_transaction_count= ('payment_value', 'count'),
                                                               distinct_payment_type_count=('payment_type', 'nunique'),
                                                               total_payment_value=('payment_value', 'sum'),
                                                               maximum_payment_installments=('payment_installments', 'max'),
                                                               payment_quality_exception_count=('flag_payment_quality_exception', 'sum')
                                                               ).reset_index()

order_payment_summary['flag_order_with_multiple_payments'] = order_payment_summary['payment_transaction_count'].gt(1)
order_payment_summary['flag_order_with_multiple_payment_types'] = order_payment_summary['distinct_payment_type_count'].gt(1)
order_payment_summary.head()

,order_id,payment_transaction_count,distinct_payment_type_count,total_payment_value,maximum_payment_installments,payment_quality_exception_count,flag_order_with_multiple_payments,flag_order_with_multiple_payment_types
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,72.19,2,0,False,False
1,00018f77f2f0320c557190d7a144bdd3,1,1,259.83,3,0,False,False
2,000229ec398224ef6ca0657da4fc703e,1,1,216.87,5,0,False,False
3,00024acbcdf0a6daa1e931b038114c75,1,1,25.78,2,0,False,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,218.04,3,0,False,False


In [12]:
# Validate the order-level payment summary and its multi-payment indicators.
print("Table shape: ", order_payment_summary.shape)
print("Duplicate order_id count: ", order_payment_summary['order_id'].duplicated().sum())
print("Missing values across the table: ", order_payment_summary.isna().sum().sum())
print("Sum of payment_transaction_count: ", order_payment_summary['payment_transaction_count'].sum())
print("Sum of total_payment_value reconciles with the payment total from delivered_order_payments: ", order_payment_summary['total_payment_value'].sum()==delivered_order_payments['payment_value'].sum())
print("Sum of payment_quality_exception_count: ",order_payment_summary['payment_quality_exception_count'].sum())
print("Number of orders flagged with multiple payment rows: ", order_payment_summary['flag_order_with_multiple_payments'].sum())
print("Maximum payment_transaction_count: ", order_payment_summary['payment_transaction_count'].max())
print("Number of orders using multiple payment types: ", order_payment_summary['flag_order_with_multiple_payment_types'].sum())
print("Every multiple-payment flag corresponds to payment_transaction_count > 1: ",(order_payment_summary['flag_order_with_multiple_payments']==order_payment_summary['payment_transaction_count'].gt(1)).all())
print("Every multiple-payment-type flag corresponds to distinct_payment_type_count > 1: ",(order_payment_summary['flag_order_with_multiple_payment_types']==order_payment_summary['distinct_payment_type_count'].gt(1)).all())
print("Number of orders with payment_quality_exception_count > 0: ", order_payment_summary['payment_quality_exception_count'].gt(0).sum())

Table shape:  (96477, 8)
Duplicate order_id count:  0
Missing values across the table:  0
Sum of payment_transaction_count:  100756
Sum of total_payment_value reconciles with the payment total from delivered_order_payments:  True
Sum of payment_quality_exception_count:  84
Number of orders flagged with multiple payment rows:  2875
Maximum payment_transaction_count:  26
Number of orders using multiple payment types:  2182
Every multiple-payment flag corresponds to payment_transaction_count > 1:  True
Every multiple-payment-type flag corresponds to distinct_payment_type_count > 1:  True
Number of orders with payment_quality_exception_count > 0:  82


### Order-Level Payment Summary Validation

The order-level payment summary contains 96,477 unique delivered orders with available payment records. All payment transactions are represented exactly once, with 100,756 transaction rows reconciling to the original delivered-order payment scope.

A total of 2,875 orders contain multiple payment transactions, while 2,182 orders use more than one payment type. The maximum number of payment transactions associated with one delivered order is 26.

The summary preserves all 84 payment-quality exception records across 82 affected orders. Both multi-payment indicators exactly match their underlying transaction-count and distinct-payment-type conditions, confirming that the order-level aggregation is internally consistent.

### 2.1 Assign a Dominant Payment Type to Each Order

Orders using a single payment type can be assigned directly to that method. For orders using multiple payment types, the dominant payment type is defined as the method contributing the largest total payment value within the order. This value-based rule allows each order to be classified once for payment-method comparisons while preserving indicators for multi-method behavior.

In [13]:
# Aggregate payment value, transaction count, and installments by order and payment type.
order_payment_type_summary = delivered_order_payments.groupby(['order_id', 'payment_type']).agg(payment_type_value=('payment_value', 'sum'),
                                                                                                payment_type_transaction_count= ('payment_type', 'count'),
                                                                                                payment_type_maximum_installments=('payment_installments', 'max')                                                                                                
                                                                                                ).reset_index()

order_payment_type_summary.head()

,order_id,payment_type,payment_type_value,payment_type_transaction_count,payment_type_maximum_installments
0,00010242fe8c5a6d1ba2dd792cb16214,credit_card,72.19,1,2
1,00018f77f2f0320c557190d7a144bdd3,credit_card,259.83,1,3
2,000229ec398224ef6ca0657da4fc703e,credit_card,216.87,1,5
3,00024acbcdf0a6daa1e931b038114c75,credit_card,25.78,1,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,credit_card,218.04,1,3


In [14]:
# validate order_payment_type_summary
print("Duplicate order_id and payment_type combinations: ", order_payment_type_summary[['order_id', 'payment_type']].duplicated().sum())
print("Missing values: ", order_payment_type_summary.isna().sum().sum())
print("Sum of payment_type_value reconciles with delivered_order_payments['payment_value']: ", np.isclose(order_payment_type_summary['payment_type_value'].sum(), delivered_order_payments['payment_value'].sum()))
print("Sum of payment_type_transaction_count: ", order_payment_type_summary['payment_type_transaction_count'].sum())
print("Distinct orders: ", order_payment_type_summary['order_id'].nunique())
print("Table row count should equal the sum of distinct_payment_type_count in order_payment_summary: ", np.isclose(len(order_payment_type_summary), order_payment_summary['distinct_payment_type_count'].sum()))
print("Minimum and maximum payment_type_transaction_count: ",
      "\nMinimum: ", order_payment_type_summary['payment_type_transaction_count'].min(),
      "\nMaximum: ", order_payment_type_summary['payment_type_transaction_count'].max())
print("Minimum and maximum payment_type_maximum_installments: ",
      "\nMinimum: ", order_payment_type_summary['payment_type_maximum_installments'].min(),
      "\nMaximum: ", order_payment_type_summary['payment_type_maximum_installments'].max())

Duplicate order_id and payment_type combinations:  0
Missing values:  0
Sum of payment_type_value reconciles with delivered_order_payments['payment_value']:  True
Sum of payment_type_transaction_count:  100756
Distinct orders:  96477
Table row count should equal the sum of distinct_payment_type_count in order_payment_summary:  True
Minimum and maximum payment_type_transaction_count:  
Minimum:  1 
Maximum:  25
Minimum and maximum payment_type_maximum_installments:  
Minimum:  0 
Maximum:  24


In [15]:
# Identify orders where multiple payment types contribute the same highest payment value.

order_payment_type_summary['maximum_payment_type_value'] = (order_payment_type_summary.groupby('order_id')['payment_type_value'].transform('max'))

# Flag payment types whose value is approximately equal to the order maximum.
order_payment_type_summary['flag_maximum_payment_type_value'] = np.isclose(order_payment_type_summary['payment_type_value'], order_payment_type_summary['maximum_payment_type_value'])

# Count the maximum-value payment types within each order.
maximum_payment_type_count_per_order = (order_payment_type_summary.groupby('order_id')['flag_maximum_payment_type_value'].sum())

# Identify orders with more than one maximum-value payment type.
tied_dominant_payment_order_ids = maximum_payment_type_count_per_order[maximum_payment_type_count_per_order.gt(1)].index

print("Number of orders with tied maximum payment-type values:", len(tied_dominant_payment_order_ids))

# Retrieve all payment-type rows belonging to tied orders.
tied_dominant_payment_orders = (order_payment_type_summary[order_payment_type_summary['order_id'].isin(tied_dominant_payment_order_ids)]
                                .sort_values(by=['order_id', 'payment_type_value', 'payment_type'],ascending=[True, False, True]).copy())

print("All tied orders use multiple payment types:", tied_dominant_payment_orders['order_id'].nunique()== tied_dominant_payment_order_ids.nunique())

display(tied_dominant_payment_orders)

Number of orders with tied maximum payment-type values: 0
All tied orders use multiple payment types: True


,order_id,payment_type,payment_type_value,payment_type_transaction_count,payment_type_maximum_installments,maximum_payment_type_value,flag_maximum_payment_type_value


#### Dominant Payment-Type Tie Validation

No delivered order contains multiple payment types with the same highest contributed payment value. Therefore, each order has exactly one value-dominant payment type, and no secondary tie-breaking rule is required.

In [16]:
# Select and rename the value-dominant payment type for each delivered order.
dominant_order_payment_type = order_payment_type_summary[order_payment_type_summary['flag_maximum_payment_type_value']==True].copy()
dominant_order_payment_type.drop(columns=['maximum_payment_type_value','flag_maximum_payment_type_value'],inplace=True)
dominant_order_payment_type.rename(columns={'payment_type': 'dominant_payment_type',
                                            'payment_type_value': 'dominant_payment_type_value',
                                            'payment_type_transaction_count': 'dominant_payment_type_transaction_count',
                                            'payment_type_maximum_installments': 'dominant_payment_type_maximum_installments'
                                            }, inplace=True)
dominant_order_payment_type.head()

,order_id,dominant_payment_type,dominant_payment_type_value,dominant_payment_type_transaction_count,dominant_payment_type_maximum_installments
0,00010242fe8c5a6d1ba2dd792cb16214,credit_card,72.19,1,2
1,00018f77f2f0320c557190d7a144bdd3,credit_card,259.83,1,3
2,000229ec398224ef6ca0657da4fc703e,credit_card,216.87,1,5
3,00024acbcdf0a6daa1e931b038114c75,credit_card,25.78,1,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,credit_card,218.04,1,3


In [17]:
# validate dominant_order_payment_type
print("dominant_order_payment_type shape: ", dominant_order_payment_type.shape)
print("Duplicate order_id count: ", dominant_order_payment_type['order_id'].duplicated().sum())
print("Missing Value: ", dominant_order_payment_type.isna().sum().sum())
print("Distinct orders: ", dominant_order_payment_type['order_id'].nunique())
print("Every selected row was originally flagged as a maximum-value payment type: ",  order_payment_type_summary.groupby('order_id')['flag_maximum_payment_type_value'].sum().eq(1).all())
print("Confirm the total number of True maximum-value flags equals the number of rows in dominant_order_payment_type: ", len(dominant_order_payment_type) == order_payment_type_summary['flag_maximum_payment_type_value'].sum())
print("Check whether every dominant_payment_type_value is greater than zero: ", dominant_order_payment_type['dominant_payment_type_value'].gt(0).all())
print("\nDisplay value counts for dominant_payment_type: ", dominant_order_payment_type['dominant_payment_type'].value_counts())

dominant_order_payment_type shape:  (96477, 5)
Duplicate order_id count:  0
Missing Value:  0
Distinct orders:  96477
Every selected row was originally flagged as a maximum-value payment type:  True
Confirm the total number of True maximum-value flags equals the number of rows in dominant_order_payment_type:  True
Check whether every dominant_payment_type_value is greater than zero:  True

Display value counts for dominant_payment_type:  dominant_payment_type
credit_card    72785
boleto         19191
voucher         3017
debit_card      1484
Name: count, dtype: int64


In [18]:
# Combine order-level payment measures with the dominant payment-type classification.
order_payment_analysis = pd.merge(order_payment_summary, dominant_order_payment_type, on="order_id", how='inner', validate='one_to_one')
order_payment_analysis.head()

,order_id,payment_transaction_count,distinct_payment_type_count,total_payment_value,maximum_payment_installments,payment_quality_exception_count,flag_order_with_multiple_payments,flag_order_with_multiple_payment_types,dominant_payment_type,dominant_payment_type_value,dominant_payment_type_transaction_count,dominant_payment_type_maximum_installments
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,72.19,2,0,False,False,credit_card,72.19,1,2
1,00018f77f2f0320c557190d7a144bdd3,1,1,259.83,3,0,False,False,credit_card,259.83,1,3
2,000229ec398224ef6ca0657da4fc703e,1,1,216.87,5,0,False,False,credit_card,216.87,1,5
3,00024acbcdf0a6daa1e931b038114c75,1,1,25.78,2,0,False,False,credit_card,25.78,1,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,218.04,3,0,False,False,credit_card,218.04,1,3


In [19]:
# validate order_payment_analysis
print("Row count compared with order_payment_summary: ", len(order_payment_analysis)==len(order_payment_summary)) 
print("Duplicate order_id count: ", order_payment_analysis['order_id'].duplicated().sum())
print("\nMissing values per column: ","\n", order_payment_analysis.isna().sum())
print("\nOrder-ID coverage between the two tables: ", (~order_payment_analysis['order_id'].isin(order_payment_summary['order_id'])).sum())

Row count compared with order_payment_summary:  True
Duplicate order_id count:  0

Missing values per column:  
 order_id                                      0
payment_transaction_count                     0
distinct_payment_type_count                   0
total_payment_value                           0
maximum_payment_installments                  0
payment_quality_exception_count               0
flag_order_with_multiple_payments             0
flag_order_with_multiple_payment_types        0
dominant_payment_type                         0
dominant_payment_type_value                   0
dominant_payment_type_transaction_count       0
dominant_payment_type_maximum_installments    0
dtype: int64

Order-ID coverage between the two tables:  0


### 2.2 Compare Orders by Dominant Payment Method

Orders are grouped by their value-dominant payment method to compare order frequency, order share, and whole-order payment values. Each payment-covered delivered order contributes to exactly one group.

Order shares use the 96,477 delivered orders with available payment records as the denominator. Payment-value comparisons use each order’s total payment value, including contributions from secondary payment methods where applicable.

In [20]:
# Summarize orders by their dominant payment method.
dominant_payment_method_summary = (order_payment_analysis.groupby('dominant_payment_type', as_index=False).agg(order_count=('order_id', 'nunique')))

# Denominator: orders with payment data in order_payment_analysis.
total_orders = order_payment_analysis['order_id'].nunique()

dominant_payment_method_summary['order_share_pct'] = (dominant_payment_method_summary['order_count'] / total_orders * 100)

dominant_payment_method_summary = (dominant_payment_method_summary.sort_values('order_count', ascending=False).reset_index(drop=True))

# Validate counts and percentage shares.
print("Order counts reconcile:", dominant_payment_method_summary['order_count'].sum() == total_orders)
print("Percentage shares total approximately 100:", abs(dominant_payment_method_summary['order_share_pct'].sum() - 100) < 1e-8)

display(dominant_payment_method_summary)

Order counts reconcile: True
Percentage shares total approximately 100: True


,dominant_payment_type,order_count,order_share_pct
0,credit_card,72785,75.442852
1,boleto,19191,19.891788
2,voucher,3017,3.127170
3,debit_card,1484,1.538190


In [21]:
# Compare whole-order payment values by dominant payment method.
dominant_payment_value_summary = (order_payment_analysis.groupby('dominant_payment_type', as_index=False).agg(
    total_order_payment_value=('total_payment_value', 'sum'),
    mean_order_payment_value=('total_payment_value', 'mean'),
    median_order_payment_value=('total_payment_value', 'median'))
    .sort_values('total_order_payment_value', ascending=False).reset_index(drop=True))

# Validate that grouped payment totals reconcile with the source.
print("Payment totals reconcile:", abs(dominant_payment_value_summary['total_order_payment_value'].sum()- order_payment_analysis['total_payment_value'].sum()) < 0.01)

# Round only for display, preserving precision in the underlying table.
display(dominant_payment_value_summary.round(2))

Payment totals reconcile: True


,dominant_payment_type,total_order_payment_value,mean_order_payment_value,median_order_payment_value
0,credit_card,12095049.25,166.18,109.41
1,boleto,2769932.58,144.33,93.78
2,voucher,349108.82,115.71,81.95
3,debit_card,208371.12,140.41,89.62


#### Dominant Payment-Method Comparison Summary

Credit card is the dominant payment method for 72,785 orders (75.44%), followed by boleto for 19,191 orders (19.89%). Voucher accounts for 3,017 orders (3.13%), while debit card accounts for 1,484 orders (1.54%).

Credit-card-dominant orders have the highest mean payment value of R$166.18 and median of R$109.41. Voucher-dominant orders have the lowest mean of R$115.71 and median of R$81.95. The mean exceeds the median in every group, suggesting that larger-value orders pull the averages upward.

Grouped order counts reconcile with all 96,477 payment-covered delivered orders, order shares total approximately 100%, and grouped payment totals reconcile with the source within R$0.01.

These comparisons describe whole-order payments grouped by dominant method, rather than amounts processed exclusively through each method. Payment values are not merchandise-only GMV, and the observed differences do not establish that payment method causes higher or lower spending.

## 3. Analyze Credit-Card Installment Behavior

This section examines installment usage in credit-card payment records associated with delivered orders and its relationship with payment value. The initial analysis uses payment-transaction grain because an order may contain multiple credit-card records with different installment counts.

Invalid installment values will remain documented but will be excluded from valid-installment comparisons. Their payment values will remain included in overall payment reconciliation.

### 3.1 Inspect the Credit-Card Installment Distribution

Credit-card payment records are selected from the validated delivered-order payment scope. The frequency of each installment count is examined before defining installment groups, with zero-installment records reported separately as known quality exceptions.

In [22]:
# Select credit-card payment records from the delivered-order payment scope.
credit_card_payments = delivered_order_payments[delivered_order_payments['payment_type'] == 'credit_card'].copy()

# Count payment records by installment value and sort by installment count.
credit_card_installment_distribution = (credit_card_payments.groupby('payment_installments').size().sort_index())

# Display the installment distribution.
display(credit_card_installment_distribution)

payment_installments
0         2
1     24759
2     12075
3     10164
4      6891
5      5095
6      3804
7      1563
8      4136
9       618
10     5150
11       22
12      128
13       15
14       14
15       72
16        5
17        7
18       27
20       16
21        3
22        1
23        1
24       18
dtype: int64

In [23]:
# Validate that the installment distribution includes every credit-card payment record.
print("Installment distribution reconciles:", credit_card_installment_distribution.sum() == len(credit_card_payments))
print("Total credit-card payment records:", len(credit_card_payments))

# Identify invalid installment counts without removing them from the original table.
invalid_installment_mask = (credit_card_payments['payment_installments'].isna() | credit_card_payments['payment_installments'].lt(1))

print("Zero-installment records:", credit_card_payments['payment_installments'].eq(0).sum())
print("Invalid-installment records:", invalid_installment_mask.sum())

# Create a separate subset for valid-installment comparisons.
valid_credit_card_payments = credit_card_payments.loc[~invalid_installment_mask].copy()

# Validate the valid subset and reconcile retained and excluded records.
print("Valid-installment records:", len(valid_credit_card_payments))
print("All retained installment counts are at least 1:", valid_credit_card_payments['payment_installments'].ge(1).all())
print("Valid and invalid record counts reconcile:", len(valid_credit_card_payments) + invalid_installment_mask.sum() == len(credit_card_payments))

Installment distribution reconciles: True
Total credit-card payment records: 74586
Zero-installment records: 2
Invalid-installment records: 2
Valid-installment records: 74584
All retained installment counts are at least 1: True
Valid and invalid record counts reconcile: True


#### Credit-Card Installment Validation Summary

The delivered-order payment scope contains 74,586 credit-card payment records. The installment distribution accounts for every record, with observed installment counts ranging from 0 to 24. A single installment is the most common value, appearing in 24,759 records.

Two records contain invalid zero-installment values. These records are excluded only from valid-installment comparisons and remain in the original payment scope for payment-value reconciliation.

The valid-installment subset contains 74,584 records, all with installment counts of at least one. Valid and invalid record counts reconcile with the original credit-card payment population. These counts represent payment records rather than distinct orders.

### 3.2 Compare Payment Values Across Installment Groups

Valid credit-card payment records are grouped into five installment bands: one installment, 2–3, 4–6, 7–12, and 13–24 installments. These descriptive bands provide a consistent structure for comparing payment frequency and payment value.

Each group is summarized using payment-record count, share of valid credit-card payment records, and mean and median payment value. Percentage shares use the 74,584 valid-installment records as the denominator.

The analysis remains at payment-transaction grain. Payment value represents the amount associated with the payment record, not the amount of each individual installment or necessarily the entire order. Differences between groups describe associations and do not establish that using more installments causes higher spending.

In [24]:
# Assign valid credit-card payment records to five ordered installment groups.
bin_edges=[0, 1, 3, 6, 12, 24]  
bin_labels=['Single installment', '2-3 installments', '4-6 installments', '7-12 installments', '13-24 installments']
valid_credit_card_payments['installment_group'] = pd.cut(valid_credit_card_payments['payment_installments'], bins=bin_edges, labels=bin_labels)

# Confirm that every valid credit-card payment record has an installment group.
print("every valid credit-card payment record has an installment group: ", valid_credit_card_payments['installment_group'].isna().sum()==0)

# Count payment records in each installment group, preserving the group order.
print("\npayment records in each installment group:","\n", valid_credit_card_payments['installment_group'].value_counts(sort=False))

# Confirm that the group counts sum to all 74,584 valid payment records.
print("\nthe group counts sum to all valid payment records: ", valid_credit_card_payments['installment_group'].value_counts(sort=False).sum()==len(valid_credit_card_payments))

every valid credit-card payment record has an installment group:  True

payment records in each installment group: 
 installment_group
Single installment    24759
2-3 installments      22239
4-6 installments      15790
7-12 installments     11617
13-24 installments      179
Name: count, dtype: int64

the group counts sum to all valid payment records:  True


In [25]:
# Summarize payment-record count, mean payment value, and median payment value by installment group.
installment_group_summary = valid_credit_card_payments.groupby('installment_group', observed=False).agg(payment_record_count=('payment_value', 'size'),
                                                                                        mean_payment_value=('payment_value', 'mean'),
                                                                                        median_payment_value=('payment_value', 'median')
                                                                                        ).reset_index()

installment_group_summary

,installment_group,payment_record_count,mean_payment_value,median_payment_value
0,Single installment,24759,95.600013,67.74
1,2-3 installments,22239,133.633245,109.89
2,4-6 installments,15790,180.479664,127.01
3,7-12 installments,11617,330.407705,204.41
4,13-24 installments,179,413.159609,252.09


In [26]:
# Calculate each group's percentage share of valid credit-card payment records.
installment_group_summary['payment_record_share_pct'] = (installment_group_summary['payment_record_count']/len(valid_credit_card_payments)) * 100

# Validate the installment-group summary and its payment-record shares.
print("installment_group_summary shape: ", installment_group_summary.shape)
print("Duplicate installment_group count: ", installment_group_summary['installment_group'].duplicated().sum())
print("Missing values across the summary: ", installment_group_summary.isna().sum().sum())
print("Sum of payment_record_count: ", installment_group_summary['payment_record_count'].sum())
print("Sum of payment_record_share_pct: ", installment_group_summary['payment_record_share_pct'].sum())
print("absolute difference of payment_record_share_pct: ", abs(installment_group_summary['payment_record_share_pct'].sum()-100)<1e-8)
print("Group order: ", "\n",installment_group_summary['installment_group'].unique())
display(installment_group_summary.round(2))

installment_group_summary shape:  (5, 5)
Duplicate installment_group count:  0
Missing values across the summary:  0
Sum of payment_record_count:  74584
Sum of payment_record_share_pct:  99.99999999999999
absolute difference of payment_record_share_pct:  True
Group order:  
 ['Single installment', '2-3 installments', '4-6 installments', '7-12 installments', '13-24 installments']
Categories (5, str): ['Single installment' < '2-3 installments' < '4-6 installments' < '7-12 installments' < '13-24 installments']


,installment_group,payment_record_count,mean_payment_value,median_payment_value,payment_record_share_pct
0,Single installment,24759,95.60,67.74,33.20
1,2-3 installments,22239,133.63,109.89,29.82
2,4-6 installments,15790,180.48,127.01,21.17
3,7-12 installments,11617,330.41,204.41,15.58
4,13-24 installments,179,413.16,252.09,0.24


#### Installment-Group Comparison Summary

The installment-group summary contains five unique groups with no missing values. Grouped payment-record counts reconcile with all 74,584 valid credit-card payment records, and percentage shares total approximately 100%. The groups retain their intended installment order.

Single-installment payments are the largest group, with 24,759 records (33.20%), followed by 2–3 installments with 22,239 records (29.82%). The 4–6 and 7–12 installment groups account for 21.17% and 15.58%, respectively. Only 179 records (0.24%) use 13–24 installments. Displayed percentages may not total exactly 100% because of rounding.

Mean and median payment values increase across the installment bands. Single-installment records have a mean payment value of R$95.60 and median of R$67.74, compared with R$413.16 and R$252.09 for the 13–24 installment group. The highest installment band contains relatively few records, so its results should be interpreted cautiously.

These results show an association between higher installment bands and larger payment values. They do not establish that additional installments cause higher spending. Payment values represent the amount associated with each credit-card payment record, not the amount of an individual installment or necessarily the entire order.

The two invalid zero-installment records remain excluded from this comparison but are retained in the original payment scope for reconciliation.

## 4. Reconcile Payment Value with Merchandise and Freight

This section compares each delivered order’s total recorded payment value with the sum of its merchandise value and freight. Item-level amounts are aggregated to order grain before joining payment information to prevent duplication.

All delivered orders remain in the reconciliation scope. The order without payment records will be identified separately rather than assigned a zero payment value. Differences will be flagged for investigation without automatically treating them as errors or removing affected orders.

### 4.1 Create Order-Level Merchandise and Freight Totals

The delivered-order-item dataset is aggregated to one row per order. Merchandise value is calculated by summing item prices, while freight value is calculated by summing item-level freight charges. Their combined value provides the comparison amount for payment reconciliation.

In [27]:
# Aggregate delivered item prices and freight charges to one row per order.
order_commercial_summary = delivered_order_items.groupby('order_id').agg(order_merchandise_value= ('price', 'sum'),
                                                                         order_freight_value= ('freight_value', 'sum')
                                                                         ).reset_index()

# Calculate the combined merchandise and freight value for each order.
order_commercial_summary['expected_order_value']= order_commercial_summary['order_merchandise_value'] + order_commercial_summary['order_freight_value']

# Validate order-level commercial totals against the delivered-item source.
print("order_commercial_summary shape: ", order_commercial_summary.shape)
print("Duplicate order_id count: ", order_commercial_summary['order_id'].duplicated().sum())
print("Missing values: ", order_commercial_summary.isna().sum().sum())
print("Distinct orders: ", order_commercial_summary['order_id'].nunique())
print("Total merchandise value reconciles: ", abs(order_commercial_summary['order_merchandise_value'].sum()-delivered_order_items['price'].sum())<0.01)
print("Total freight value reconciles: ", abs(order_commercial_summary['order_freight_value'].sum()-delivered_order_items['freight_value'].sum())<0.01)
print("Combined value is correct for every order:", ((order_commercial_summary['expected_order_value']- (order_commercial_summary['order_merchandise_value']+ order_commercial_summary['order_freight_value'])).abs() < 0.01).all())
display(order_commercial_summary.head())

order_commercial_summary shape:  (96478, 4)
Duplicate order_id count:  0
Missing values:  0
Distinct orders:  96478
Total merchandise value reconciles:  True
Total freight value reconciles:  True
Combined value is correct for every order: True


,order_id,order_merchandise_value,order_freight_value,expected_order_value
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04


#### Order-Level Commercial Summary Validation

The commercial summary contains 96,478 unique delivered orders and four columns, with no duplicate order IDs or missing values. Aggregated merchandise and freight totals reconcile with the delivered-item source within R$0.01.

For every order, the expected order value equals merchandise value plus freight within R$0.01. The table is ready to be joined with order-level payment information while preserving the complete delivered-order population.

### 4.2 Join Commercial Totals with Payment Information

Commercial totals are left-joined with the order-level payment analysis table using a one-to-one relationship on order_id. This preserves all 96,478 delivered orders and adds payment measures and dominant payment-type classifications where available.

A merge indicator distinguishes orders with payment information from the known order without payment records. Missing payment values remain missing rather than being replaced with zero.

In [28]:
# Left-join order-level commercial totals with payment information and retain a coverage indicator.
order_payment_reconciliation = pd.merge(order_commercial_summary, order_payment_analysis, on='order_id', how='left',  validate='one_to_one',indicator='payment_merge_status')

# Validate join cardinality, payment coverage, and preservation of payment totals.
print("order_payment_reconciliation shape: ", order_payment_reconciliation.shape)
print("Duplicate order_id count: ", order_payment_reconciliation['order_id'].duplicated().sum())
print("\nMerge indicator count: ", order_payment_reconciliation['payment_merge_status'].value_counts())
print("\nMissing commercial values: ", "\n",order_payment_reconciliation[order_commercial_summary.columns].isna().sum())
print("Total payment value reconciliation:", abs(order_payment_reconciliation['total_payment_value'].sum()- order_payment_analysis['total_payment_value'].sum()) < 0.01)

order_payment_reconciliation shape:  (96478, 16)
Duplicate order_id count:  0

Merge indicator count:  payment_merge_status
both          96477
left_only         1
right_only        0
Name: count, dtype: int64

Missing commercial values:  
 order_id                   0
order_merchandise_value    0
order_freight_value        0
expected_order_value       0
dtype: int64
Total payment value reconciliation: True


In [29]:
print("Missing total_payment_value: ","\n", order_payment_reconciliation[order_payment_analysis.columns].isna().sum())

# Display the delivered order without matching payment information.
order_payment_reconciliation[order_payment_reconciliation['payment_merge_status']=='left_only']

Missing total_payment_value:  
 order_id                                      0
payment_transaction_count                     1
distinct_payment_type_count                   1
total_payment_value                           1
maximum_payment_installments                  1
payment_quality_exception_count               1
flag_order_with_multiple_payments             1
flag_order_with_multiple_payment_types        1
dominant_payment_type                         1
dominant_payment_type_value                   1
dominant_payment_type_transaction_count       1
dominant_payment_type_maximum_installments    1
dtype: int64


,order_id,order_merchandise_value,order_freight_value,expected_order_value,payment_transaction_count,distinct_payment_type_count,total_payment_value,maximum_payment_installments,payment_quality_exception_count,flag_order_with_multiple_payments,flag_order_with_multiple_payment_types,dominant_payment_type,dominant_payment_type_value,dominant_payment_type_transaction_count,dominant_payment_type_maximum_installments,payment_merge_status
72234,bfbd0f9bdef84302105ad712db648a6c,134.97,8.49,143.46,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


#### Commercial and Payment Join Validation Summary

The reconciliation table contains 96,478 unique delivered orders and 16 columns, with no duplicate order IDs or missing commercial values. Payment information matches 96,477 orders, and the merged payment total reconciles with the original order-level payment analysis within R$0.01.

The remaining order, bfbd0f9bdef84302105ad712db648a6c, has merchandise value of R$134.97 and freight of R$8.49, giving an expected order value of R$143.46. Its payment information is unavailable and remains missing rather than being replaced with zero.

This order is retained in the delivered-order population but cannot be assessed for agreement between recorded payments and merchandise plus freight.

### 4.3 Calculate Order-Level Payment Differences

Payment difference is calculated as total recorded payment value minus expected order value. Positive differences indicate recorded payments above merchandise plus freight, while negative differences indicate recorded payments below that amount.

The absolute difference measures the size of the discrepancy regardless of direction. Orders without payment information retain missing differences and are assessed separately.

In [30]:
# Calculate the signed difference between recorded payment and expected order value.
order_payment_reconciliation['payment_difference'] = order_payment_reconciliation['total_payment_value'] - order_payment_reconciliation['expected_order_value'] 

# Calculate the absolute payment difference for each order.
order_payment_reconciliation['absolute_payment_difference'] = order_payment_reconciliation['payment_difference'].abs()

# Validate payment-difference coverage and inspect the results.
print("order_payment_reconciliation shape: ", order_payment_reconciliation.shape)
print("\nMissing differences: ",
      "\n", "payment_difference: ", order_payment_reconciliation['payment_difference'].isna().sum(),
      "\n", "absolute_payment_difference: ", order_payment_reconciliation['absolute_payment_difference'].isna().sum())

print("\nNon-missing differences: ",
      "\n", "payment_difference: ", order_payment_reconciliation['payment_difference'].notna().sum(),
      "\n", "absolute_payment_difference: ", order_payment_reconciliation['absolute_payment_difference'].notna().sum())

display(order_payment_reconciliation[order_payment_reconciliation['payment_difference'].isna() | order_payment_reconciliation['absolute_payment_difference'].isna()])

# Confirm that missing differences occur exactly for orders without payment information.
missing_payment_mask = order_payment_reconciliation['payment_merge_status'].eq('left_only')

print("Missing payment_difference aligns with missing payment information:",(order_payment_reconciliation['payment_difference'].isna() == missing_payment_mask).all())

print("Missing absolute_payment_difference aligns with missing payment information:",(order_payment_reconciliation['absolute_payment_difference'].isna() == missing_payment_mask).all())

# Confirm that all non-missing absolute payment differences are non-negative.
print("All non-missing absolute differences are non-negative:", order_payment_reconciliation['absolute_payment_difference'].dropna().ge(0).all())

# Display the payment totals, expected values, and differences for the first five orders.
display(order_payment_reconciliation[['order_id', 'total_payment_value', 'expected_order_value', 'payment_difference', 'absolute_payment_difference']].head())



order_payment_reconciliation shape:  (96478, 18)

Missing differences:  
 payment_difference:  1 
 absolute_payment_difference:  1

Non-missing differences:  
 payment_difference:  96477 
 absolute_payment_difference:  96477


,order_id,order_merchandise_value,order_freight_value,expected_order_value,payment_transaction_count,distinct_payment_type_count,total_payment_value,maximum_payment_installments,payment_quality_exception_count,flag_order_with_multiple_payments,flag_order_with_multiple_payment_types,dominant_payment_type,dominant_payment_type_value,dominant_payment_type_transaction_count,dominant_payment_type_maximum_installments,payment_merge_status,payment_difference,absolute_payment_difference
72234,bfbd0f9bdef84302105ad712db648a6c,134.97,8.49,143.46,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,NaN,NaN


Missing payment_difference aligns with missing payment information: True
Missing absolute_payment_difference aligns with missing payment information: True
All non-missing absolute differences are non-negative: True


,order_id,total_payment_value,expected_order_value,payment_difference,absolute_payment_difference
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,72.19,0.000000e+00,0.000000e+00
1,00018f77f2f0320c557190d7a144bdd3,259.83,259.83,0.000000e+00,0.000000e+00
2,000229ec398224ef6ca0657da4fc703e,216.87,216.87,0.000000e+00,0.000000e+00
3,00024acbcdf0a6daa1e931b038114c75,25.78,25.78,0.000000e+00,0.000000e+00
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,218.04,-2.842171e-14,2.842171e-14


#### Payment-Difference Validation Summary

Payment differences are available for 96,477 of the 96,478 delivered orders. Both difference columns remain missing only for the known order without payment information, and all non-missing absolute differences are non-negative.

Small residual differences, such as -2.842171e-14, arise from floating-point arithmetic. These should not be interpreted as monetary discrepancies. The original differences are preserved, while reconciliation classification will use differences rounded to two decimal places.

### 4.4 Classify Payment Reconciliation Results

Orders are classified as matched, payment above expected, payment below expected, or missing payment information. Classification uses payment differences rounded to two decimal places so that floating-point residuals do not create false discrepancies.

A rounded difference of zero is classified as matched. Positive and negative differences indicate recorded payment above or below merchandise plus freight, respectively. These labels describe differences between the recorded amounts and do not establish their underlying cause.

In [31]:
# Round payment differences to cents for classification while preserving the original values.
rounded_payment_difference = order_payment_reconciliation['payment_difference'].round(2)

# Assign a reconciliation status using payment availability and the rounded difference.
conditions = [ order_payment_reconciliation['payment_merge_status'].eq('left_only'), rounded_payment_difference.eq(0), rounded_payment_difference.gt(0), rounded_payment_difference.lt(0)]

status_labels = ['missing_payment', 'matched', 'payment_above_expected', 'payment_below_expected']

order_payment_reconciliation['payment_reconciliation_status'] = np.select(conditions, status_labels, default='unclassified')

In [32]:
# Validate reconciliation-status coverage and display status counts.
status = order_payment_reconciliation['payment_reconciliation_status']
status_counts = status.value_counts()

print("Table shape:", order_payment_reconciliation.shape)
print("Missing status labels:", status.isna().sum())
print("Unexpected or fallback labels:", (~status.isin(status_labels)).sum())
print("Missing-payment count:", status.eq('missing_payment').sum())

print("Other three status counts combined:", status.isin(['matched', 'payment_above_expected', 'payment_below_expected']).sum())

print("All status counts combined:", status_counts.sum())
print("Status counts reconcile:", status_counts.sum() == len(order_payment_reconciliation))

print("\nReconciliation-status counts:")
display(status_counts)

Table shape: (96478, 19)
Missing status labels: 0
Unexpected or fallback labels: 0
Missing-payment count: 1
Other three status counts combined: 96477
All status counts combined: 96478
Status counts reconcile: True

Reconciliation-status counts:


payment_reconciliation_status
matched                   95908
payment_above_expected      362
payment_below_expected      207
missing_payment               1
Name: count, dtype: int64

#### Payment Reconciliation Classification Summary

Of the 96,477 delivered orders with payment information, 95,908 (99.41%) have recorded payment totals that match merchandise plus freight after rounding the difference to two decimal places.

A total of 569 orders (0.59%) have non-zero differences: 362 have recorded payments above the expected value, while 207 have recorded payments below it. One additional delivered order has no payment information and remains classified separately as missing_payment.

All 96,478 delivered orders have an expected reconciliation status, with no missing or unclassified labels. These discrepancies require further investigation; their direction alone does not establish an error, overcharge, refund, or loss.

### 4.5 Inspect Payment Discrepancy Magnitudes

Orders with recorded payments above or below merchandise plus freight are examined to quantify the size of the differences. The analysis separates discrepancy direction and identifies the largest absolute differences for further investigation.

All flagged orders remain in the dataset. The order without payment information is excluded from discrepancy-magnitude calculations because its payment difference is unknown.

In [33]:
# Select orders with payment discrepancies while excluding matched and missing-payment orders.
payment_discrepancy_orders = order_payment_reconciliation.loc[order_payment_reconciliation['payment_reconciliation_status'].isin(['payment_above_expected', 'payment_below_expected'])].copy()

# Validate the discrepancy subset before analyzing difference sizes.
print("Discrepancy table shape:", payment_discrepancy_orders.shape)

print("Duplicate order_id count:", payment_discrepancy_orders['order_id'].duplicated().sum())

print("\nMissing values in difference columns:\n", payment_discrepancy_orders[['payment_difference', 'absolute_payment_difference']].isna().sum())

print("\nDiscrepancy status counts:\n", payment_discrepancy_orders['payment_reconciliation_status'].value_counts())

print("\nEvery rounded absolute difference is positive:", payment_discrepancy_orders['absolute_payment_difference'].round(2).gt(0).all())

Discrepancy table shape: (569, 19)
Duplicate order_id count: 0

Missing values in difference columns:
 payment_difference             0
absolute_payment_difference    0
dtype: int64

Discrepancy status counts:
 payment_reconciliation_status
payment_above_expected    362
payment_below_expected    207
Name: count, dtype: int64

Every rounded absolute difference is positive: True


In [34]:
# Summarize discrepancy counts and absolute difference amounts by reconciliation status.
payment_discrepancy_summary = payment_discrepancy_orders.groupby('payment_reconciliation_status', as_index=False).agg(
    order_count=('order_id', 'nunique'),
    total_absolute_difference=('absolute_payment_difference', 'sum'),
    median_absolute_difference=('absolute_payment_difference', 'median'),
    maximum_absolute_difference=('absolute_payment_difference', 'max')
    )

# Validate summary counts and reconcile total absolute differences with the discrepancy subset.
print("Summary shape:", payment_discrepancy_summary.shape)

print("Duplicate statuses:", payment_discrepancy_summary['payment_reconciliation_status'].duplicated().sum())

print("Missing values:", payment_discrepancy_summary.isna().sum().sum())

print("Order-count sum:", payment_discrepancy_summary['order_count'].sum())

print("Order counts reconcile:", payment_discrepancy_summary['order_count'].sum() == len(payment_discrepancy_orders))

print("Total absolute differences reconcile:", abs(payment_discrepancy_summary['total_absolute_difference'].sum()- payment_discrepancy_orders['absolute_payment_difference'].sum()) < 0.01)

# Display the discrepancy summary rounded to two decimal places.
display(payment_discrepancy_summary.round(2))

Summary shape: (2, 5)
Duplicate statuses: 0
Missing values: 0
Order-count sum: 569
Order counts reconcile: True
Total absolute differences reconcile: True


,payment_reconciliation_status,order_count,total_absolute_difference,median_absolute_difference,maximum_absolute_difference
0,payment_above_expected,362,3032.24,3.44,182.81
1,payment_below_expected,207,200.76,0.01,51.62


#### Payment Discrepancy Magnitude Summary

The discrepancy summary covers 569 unique orders, and its counts and total absolute differences reconcile with the discrepancy subset.

The 362 orders with payments above the expected value account for R$3,032.24 in absolute differences, with a median of R$3.44 and a maximum of R$182.81. The 207 orders with payments below the expected value account for R$200.76, with a median of R$0.01 and a maximum of R$51.62.

The combined absolute discrepancy is R$3,233.00. Above-expected payments account for most of this amount. The below-expected median indicates that small differences are common in that group, although larger discrepancies also exist.

These amounts describe differences between recorded payments and merchandise plus freight. They do not establish confirmed overcharges, refunds, revenue gains, or losses.

In [35]:
# Select the ten orders with the largest absolute payment differences.
top_payment_discrepancies = (payment_discrepancy_orders.sort_values('absolute_payment_difference', ascending=False).head(10).copy())

# Display commercial values, payment differences, and payment-quality indicators for investigation.
display(top_payment_discrepancies[
        [
            'order_id',
            'expected_order_value',
            'total_payment_value',
            'payment_difference',
            'absolute_payment_difference',
            'payment_reconciliation_status',
            'payment_transaction_count',
            'distinct_payment_type_count',
            'payment_quality_exception_count'
        ]].round(2))

,order_id,expected_order_value,total_payment_value,payment_difference,absolute_payment_difference,payment_reconciliation_status,payment_transaction_count,distinct_payment_type_count,payment_quality_exception_count
77799,ce6d150fb29ada17d2082f4847107665,1403.66,1586.47,182.81,182.81,payment_above_expected,1.0,1.0,0.0
41570,6e5fe7366a2e1bfbf3257dba0af1267f,287.91,406.92,119.01,119.01,payment_above_expected,1.0,1.0,0.0
42467,70b742795bc441e94a44a084b6d9ce7a,466.93,578.82,111.89,111.89,payment_above_expected,1.0,1.0,0.0
57461,996c7e73600ad3723e8627ab7bef81e4,587.90,664.43,76.53,76.53,payment_above_expected,1.0,1.0,0.0
42470,70b7e94ea46d3e8b5bc12a50186edaf0,213.15,274.84,61.69,61.69,payment_above_expected,1.0,1.0,0.0
70919,bc2c82b0ef78d2252b6176d1972db7c9,242.01,303.02,61.01,61.01,payment_above_expected,1.0,1.0,0.0
65989,af9ffff2ce6b3defd34fd4c78857a379,413.17,466.97,53.80,53.80,payment_above_expected,1.0,1.0,0.0
14320,262118ce178bb3e4590a3adcf6d62e6b,177.74,126.12,-51.62,51.62,payment_below_expected,1.0,1.0,0.0
72286,bfdb5bbb06458d600a33d61f5f287472,348.93,394.36,45.43,45.43,payment_above_expected,1.0,1.0,0.0
53097,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,254.45,293.89,39.44,39.44,payment_above_expected,1.0,1.0,0.0


#### Largest Payment Discrepancies

Among the ten orders with the largest absolute payment differences, nine have recorded payments above merchandise plus freight and one has recorded payment below that amount. The largest positive difference is R$182.81, while the largest negative difference is R$51.62.

All ten orders contain a single payment transaction and a single payment type, with no existing payment-quality exceptions. Multiple payment records therefore do not explain these particular discrepancies. The existing quality flags also do not capture these cross-table value differences.

The underlying causes remain unconfirmed. These orders are retained and flagged for reconciliation rather than automatically corrected or removed.

### 4.6 Compare Discrepancies with Payment Characteristics

Reconciliation outcomes are compared with multiple-payment behavior and existing payment-quality exceptions. Both matched and discrepant orders are included so that discrepancy rates can be compared against the relevant order populations.

The order without payment information is excluded because its reconciliation outcome cannot be assessed. Any observed differences between groups describe associations rather than causes.

In [36]:
# Select payment-covered orders for comparison of reconciliation outcomes.
payment_reconciliation_profile = order_payment_reconciliation.loc[order_payment_reconciliation['payment_merge_status'].eq('both')].copy()

# Flag payment discrepancies and orders with existing payment-quality exceptions.
payment_reconciliation_profile['flag_payment_discrepancy'] = (payment_reconciliation_profile['payment_reconciliation_status'].isin(['payment_above_expected', 'payment_below_expected']))

payment_reconciliation_profile['flag_any_payment_quality_exception'] = (payment_reconciliation_profile['payment_quality_exception_count'].gt(0))

In [37]:
# Validate the profiling population and flag counts.
print("Profile shape:", payment_reconciliation_profile.shape)

print("Duplicate order IDs:", payment_reconciliation_profile['order_id'].duplicated().sum())

print("\nMissing values in new flags:\n", payment_reconciliation_profile[['flag_payment_discrepancy', 'flag_any_payment_quality_exception']].isna().sum())

print("\nDiscrepancy flag count:", payment_reconciliation_profile['flag_payment_discrepancy'].sum())

print("Non-discrepant count:", (~payment_reconciliation_profile['flag_payment_discrepancy']).sum())

print("Payment-quality exception flag count:", payment_reconciliation_profile['flag_any_payment_quality_exception'].sum())

Profile shape: (96477, 21)
Duplicate order IDs: 0

Missing values in new flags:
 flag_payment_discrepancy              0
flag_any_payment_quality_exception    0
dtype: int64

Discrepancy flag count: 569
Non-discrepant count: 95908
Payment-quality exception flag count: 82


In [38]:
# Compare discrepancy counts for single-payment and multiple-payment orders.
multiple_payment_discrepancy_summary = payment_reconciliation_profile.groupby('flag_order_with_multiple_payments', as_index=False).agg(order_count=('order_id', 'nunique'),
                                                                                                                                       discrepancy_order_count=('flag_payment_discrepancy', 'sum')
                                                                                                                                       )

# Calculate the discrepancy rate within each payment group.
multiple_payment_discrepancy_summary['discrepancy_rate_pct'] = (multiple_payment_discrepancy_summary['discrepancy_order_count']/ multiple_payment_discrepancy_summary['order_count'] * 100)

# Validate group counts and discrepancy totals.
print("Summary shape:", multiple_payment_discrepancy_summary.shape)

print("Duplicate grouping values:", multiple_payment_discrepancy_summary['flag_order_with_multiple_payments'].duplicated().sum())

print("Missing values:", multiple_payment_discrepancy_summary.isna().sum().sum())

print("\nOrder counts by group:\n",multiple_payment_discrepancy_summary.set_index('flag_order_with_multiple_payments')['order_count'])

print("\nTotal order count:", multiple_payment_discrepancy_summary['order_count'].sum())

print("Order counts reconcile:", multiple_payment_discrepancy_summary['order_count'].sum() == len(payment_reconciliation_profile))

print("Total discrepancy count:", multiple_payment_discrepancy_summary['discrepancy_order_count'].sum())

print("Discrepancy counts reconcile:", multiple_payment_discrepancy_summary['discrepancy_order_count'].sum() == payment_reconciliation_profile['flag_payment_discrepancy'].sum())

print("Discrepancy rates are between 0 and 100:", multiple_payment_discrepancy_summary['discrepancy_rate_pct'].between(0, 100).all())

# Display the comparison rounded to two decimal places.
display(multiple_payment_discrepancy_summary.round(2))

Summary shape: (2, 4)
Duplicate grouping values: 0
Missing values: 0

Order counts by group:
 flag_order_with_multiple_payments
False    93602
True      2875
Name: order_count, dtype: int64

Total order count: 96477
Order counts reconcile: True
Total discrepancy count: 569
Discrepancy counts reconcile: True
Discrepancy rates are between 0 and 100: True


,flag_order_with_multiple_payments,order_count,discrepancy_order_count,discrepancy_rate_pct
0,False,93602,543,0.58
1,True,2875,26,0.90


#### Payment Discrepancies by Transaction Count

Single-payment orders contain 543 discrepancies across 93,602 orders, giving a discrepancy rate of 0.58%. Multiple-payment orders contain 26 discrepancies across 2,875 orders, giving a rate of 0.90%.

The observed discrepancy rate is approximately 0.32 percentage points higher among multiple-payment orders. However, most discrepant orders have only one payment transaction, so multiple-payment behavior does not explain the majority of discrepancies.

Group counts reconcile with all 96,477 payment-covered orders, and discrepancy counts reconcile with the 569 identified cases. This comparison describes an association and does not establish that multiple payments cause discrepancies.

In [39]:
# Compare discrepancy counts for orders with and without existing payment-quality exceptions.
quality_exception_discrepancy_summary = payment_reconciliation_profile.groupby('flag_any_payment_quality_exception', as_index=False).agg(order_count=('order_id', 'nunique'),
                                                                                                                                         discrepancy_order_count=('flag_payment_discrepancy', 'sum'))

# Calculate the discrepancy rate within each quality-exception group.
quality_exception_discrepancy_summary['discrepancy_rate_pct'] = (quality_exception_discrepancy_summary['discrepancy_order_count']/ quality_exception_discrepancy_summary['order_count'] * 100)

# Validate group counts and discrepancy totals.
print("Summary shape:", quality_exception_discrepancy_summary.shape)

print("Duplicate grouping values:", quality_exception_discrepancy_summary['flag_any_payment_quality_exception'].duplicated().sum())

print("Missing values:", quality_exception_discrepancy_summary.isna().sum().sum())

print("\nOrder counts by group:\n", quality_exception_discrepancy_summary.set_index('flag_any_payment_quality_exception')['order_count'])

print("\nTotal order count:", quality_exception_discrepancy_summary['order_count'].sum())

print("Order counts reconcile:", quality_exception_discrepancy_summary['order_count'].sum() == len(payment_reconciliation_profile))

print("Total discrepancy count:", quality_exception_discrepancy_summary['discrepancy_order_count'].sum())

print("Discrepancy counts reconcile:", quality_exception_discrepancy_summary['discrepancy_order_count'].sum() == payment_reconciliation_profile['flag_payment_discrepancy'].sum())

print("Discrepancy rates are between 0 and 100:", quality_exception_discrepancy_summary['discrepancy_rate_pct'].between(0, 100).all())

# Display the comparison rounded to two decimal places.
display(quality_exception_discrepancy_summary.round(2))

Summary shape: (2, 4)
Duplicate grouping values: 0
Missing values: 0

Order counts by group:
 flag_any_payment_quality_exception
False    96395
True        82
Name: order_count, dtype: int64

Total order count: 96477
Order counts reconcile: True
Total discrepancy count: 569
Discrepancy counts reconcile: True
Discrepancy rates are between 0 and 100: True


,flag_any_payment_quality_exception,order_count,discrepancy_order_count,discrepancy_rate_pct
0,False,96395,567,0.59
1,True,82,2,2.44


#### Payment Discrepancies by Existing Quality Exceptions

Orders without existing payment-quality exceptions contain 567 discrepancies across 96,395 orders, giving a discrepancy rate of 0.59%. Orders with existing exceptions contain two discrepancies across 82 orders, giving a rate of 2.44%.

The flagged group has a higher observed discrepancy rate, but its small size limits the strength of this comparison. Existing payment-quality flags overlap with only two of the 569 reconciliation discrepancies. Transaction-level quality checks and cross-table payment reconciliation therefore identify different issues and should be retained separately.

Group counts reconcile with all 96,477 payment-covered orders, and discrepancy counts reconcile with all 569 identified cases. These findings describe associations and do not establish the causes of the payment differences.

## 5. Analyze Order Merchandise-Value Bands

This section compares delivered orders across merchandise-value bands. Order merchandise value excludes freight and represents the combined price of all items within an order, rather than the price of an individual product.

All 96,478 delivered orders remain in scope, including the order without payment information. The merchandise-value distribution is inspected before selecting descriptive band boundaries.

### 5.1 Inspect the Order Merchandise-Value Distribution

Summary statistics and selected percentiles are examined to understand typical order values, the spread of values, and the upper tail. These results will guide the selection of interpretable bands without automatically excluding unusually large orders.

In [40]:
# Inspect the distribution of order merchandise values before defining value bands.
display(order_commercial_summary['order_merchandise_value'].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

# Validate merchandise-value completeness and identify zero or negative values.
print("Order count:", len(order_commercial_summary))

print("Missing merchandise values:", order_commercial_summary['order_merchandise_value'].isna().sum())

print("Zero merchandise values:", order_commercial_summary['order_merchandise_value'].eq(0).sum())

print("Negative merchandise values:", order_commercial_summary['order_merchandise_value'].lt(0).sum())

count    96478.00
mean       137.04
std        209.05
min          0.85
25%         45.90
50%         86.57
75%        149.90
90%        269.00
95%        399.00
99%        990.00
max      13440.00
Name: order_merchandise_value, dtype: float64

Order count: 96478
Missing merchandise values: 0
Zero merchandise values: 0
Negative merchandise values: 0


#### Order Merchandise-Value Distribution Summary

All 96,478 delivered orders have complete, positive merchandise values, with no zero or negative amounts. Values range from R$0.85 to R$13,440.00.

The median order merchandise value is R$86.57, compared with a mean of R$137.04. The 75th percentile is R$149.90, the 95th percentile is R$399.00, and the 99th percentile is R$990.00. These results indicate a right-skewed distribution, with a relatively small number of high-value orders extending the upper tail.

High-value orders remain in the analysis. Descriptive bands will use rounded monetary boundaries informed by the distribution rather than treating high values as errors.

### 5.2 Assign Order Merchandise-Value Bands

Orders are assigned to six merchandise-value bands: up to R$50, above R$50 to R$100, above R$100 to R$150, above R$150 to R$400, above R$400 to R$1,000, and above R$1,000.

These boundaries provide interpretable comparisons informed by the observed distribution. They are descriptive reporting bands, not equal-sized groups or externally established customer segments. Every delivered order is assigned to exactly one band.

In [41]:
# Assign each delivered order to an ordered merchandise-value band.
order_value_bin_edges = [0, 50, 100, 150, 400, 1000, np.inf]
order_value_bin_labels = ['0-50', '50-100', '100-150', '150-400', '400-1000', '1000+']

order_commercial_summary['order_value_band'] = pd.cut(order_commercial_summary['order_merchandise_value'],
                                                      bins=order_value_bin_edges,
                                                      labels=order_value_bin_labels,
                                                      right=True)

# Validate band coverage and display order counts in band order.
print("Table shape:", order_commercial_summary.shape)

print("Missing order-value bands:", order_commercial_summary['order_value_band'].isna().sum())

print("Duplicate order IDs:", order_commercial_summary['order_id'].duplicated().sum())

order_value_band_counts = (order_commercial_summary['order_value_band'].value_counts(sort=False))

print("\nOrder counts by value band:")
display(order_value_band_counts)

print("Sum of band counts:", order_value_band_counts.sum())

print("Band counts reconcile:", order_value_band_counts.sum() == len(order_commercial_summary))

Table shape: (96478, 5)
Missing order-value bands: 0
Duplicate order IDs: 0

Order counts by value band:


order_value_band
0-50        29045
50-100      27606
100-150     16501
150-400     18675
400-1000     3746
1000+         905
Name: count, dtype: int64

Sum of band counts: 96478
Band counts reconcile: True


In [42]:
# Summarize delivered-order counts and merchandise GMV by order-value band.
order_value_band_summary = order_commercial_summary.groupby('order_value_band', as_index=False, observed=False).agg(order_count=('order_id', 'nunique'),merchandise_gmv=('order_merchandise_value', 'sum'))

# Calculate each band's share of delivered orders and merchandise GMV.
total_delivered_orders = order_commercial_summary['order_id'].nunique()
total_merchandise_gmv = order_commercial_summary['order_merchandise_value'].sum()

order_value_band_summary['order_share_pct'] = (order_value_band_summary['order_count'] / total_delivered_orders * 100)

order_value_band_summary['gmv_share_pct'] = (order_value_band_summary['merchandise_gmv'] / total_merchandise_gmv * 100)

In [43]:
# Validate summary completeness, order counts, GMV totals, and percentage shares.
print("Summary shape:", order_value_band_summary.shape)

print("Duplicate bands:", order_value_band_summary['order_value_band'].duplicated().sum())

print("Missing values:", order_value_band_summary.isna().sum().sum())
print("Total order count:", order_value_band_summary['order_count'].sum())

print("Order counts reconcile:", order_value_band_summary['order_count'].sum() == total_delivered_orders)

print("Merchandise GMV reconciles:", abs(order_value_band_summary['merchandise_gmv'].sum() - total_merchandise_gmv) < 0.01)

print("Order shares total approximately 100:", abs(order_value_band_summary['order_share_pct'].sum() - 100) < 1e-8)

print("GMV shares total approximately 100:", abs(order_value_band_summary['gmv_share_pct'].sum() - 100) < 1e-8)

# Display the band summary in category order, rounded only for presentation.
display(order_value_band_summary.sort_values('order_value_band').round(2))

Summary shape: (6, 5)
Duplicate bands: 0
Missing values: 0
Total order count: 96478
Order counts reconcile: True
Merchandise GMV reconciles: True
Order shares total approximately 100: True
GMV shares total approximately 100: True


,order_value_band,order_count,merchandise_gmv,order_share_pct,gmv_share_pct
0,0-50,29045,929291.71,30.11,7.03
1,50-100,27606,2092612.98,28.61,15.83
2,100-150,16501,2086383.49,17.10,15.78
3,150-400,18675,4335987.80,19.36,32.79
4,400-1000,3746,2312659.72,3.88,17.49
5,1000+,905,1464562.41,0.94,11.08


#### Order Merchandise-Value Band Summary

The six merchandise-value bands cover all 96,478 delivered orders, with no duplicate bands or missing summary values. Order counts and merchandise GMV reconcile with the source, and both percentage-share columns total approximately 100%.

Orders valued up to R$50 form the largest group by count, representing 30.11% of orders but only 7.03% of merchandise GMV. The above-R$150-to-R$400 band contributes the largest share of merchandise GMV at 32.79%, from 19.36% of orders.

Orders above R$400 account for approximately 4.82% of orders and 28.57% of merchandise GMV. Within this population, the 905 orders above R$1,000 contribute 11.08% of merchandise GMV despite representing only 0.94% of orders.

These results describe the concentration of merchandise value across order sizes. They do not measure profitability, individual product prices, or customer lifetime value.

## 6. Analyze Freight Burden

Freight burden is defined as order freight value divided by order merchandise value, expressed as a percentage. It measures the size of the freight charge relative to the merchandise purchased, rather than freight as a share of the combined order total.

All 96,478 delivered orders have positive merchandise values, so the ratio can be calculated for the complete population. Zero-freight orders remain included, and unusually high ratios will be inspected rather than automatically removed.

### 6.1 Calculate and Inspect Order-Level Freight Burden

The freight-burden distribution is examined before selecting reporting bands. A burden of 50% means that freight equals half the merchandise value, while a burden above 100% means freight exceeds merchandise value.

In [44]:
# Calculate freight as a percentage of each order's merchandise value.
order_commercial_summary['freight_burden_pct'] = (order_commercial_summary['order_freight_value']/ order_commercial_summary['order_merchandise_value'] * 100)

# Validate freight-burden completeness and inspect its distribution.
print("Table shape:", order_commercial_summary.shape)

print("Missing freight-burden values:", order_commercial_summary['freight_burden_pct'].isna().sum())

print("Infinite freight-burden values:", np.isinf(order_commercial_summary['freight_burden_pct']).sum())

print("Negative freight-burden values:", order_commercial_summary['freight_burden_pct'].lt(0).sum())

print("Zero-freight orders:", order_commercial_summary['order_freight_value'].eq(0).sum())

print("Orders with freight burden above 100%:", order_commercial_summary['freight_burden_pct'].gt(100).sum())

print("\nFreight-burden distribution (%):")
display(order_commercial_summary['freight_burden_pct'].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

Table shape: (96478, 6)
Missing freight-burden values: 0
Infinite freight-burden values: 0
Negative freight-burden values: 0
Zero-freight orders: 336
Orders with freight burden above 100%: 3075

Freight-burden distribution (%):


count    96478.00
mean        30.84
std         31.16
min          0.00
25%         13.20
50%         22.44
75%         38.05
90%         62.23
95%         83.89
99%        145.85
max       2144.71
Name: freight_burden_pct, dtype: float64

#### Order-Level Freight-Burden Validation Summary

Freight burden is available for all 96,478 delivered orders, with no missing, infinite, or negative values. There are 336 zero-freight orders, which remain included in the analysis.

The median freight burden is 22.44%, while the mean is 30.84%. The 75th percentile is 38.05%, and the 95th percentile is 83.89%. These statistics describe individual order ratios; the mean is not the same as total freight divided by total merchandise value.

For 3,075 orders, freight exceeds merchandise value. The maximum burden is 2,144.71%, indicating an extreme ratio that requires inspection alongside its underlying merchandise and freight amounts. High ratios are retained and are not automatically treated as errors.

In [45]:
# Select the ten orders with the highest freight burden.
highest_freight_burden_orders = order_commercial_summary.sort_values('freight_burden_pct', ascending=False).head(10).copy()

print("Selected order count:", len(highest_freight_burden_orders))

# Display merchandise and freight amounts alongside the ratio to inspect extreme values.
display(highest_freight_burden_orders[['order_id','order_merchandise_value','order_freight_value','expected_order_value','freight_burden_pct']].round(2))

Selected order count: 10


,order_id,order_merchandise_value,order_freight_value,expected_order_value,freight_burden_pct
41616,6e864b3f0ec71031117ad4cf46b7f2a1,0.85,18.23,19.08,2144.71
23648,3ee6513ae7ea23bdfab5b9ab60bffcb5,0.85,18.23,19.08,2144.71
46882,7c92284adbec8033d160303d83065dbf,9.18,54.69,63.87,595.75
67274,b2d30dfaf7114de8ab3f737d062032fe,5.95,34.15,40.10,573.95
12803,21fb84496c034392ca2aafe7f7919a3f,6.99,37.90,44.89,542.20
60531,a16607f4220f0e60ab4c71b915dd380c,14.00,74.08,88.08,529.14
56117,95d6357ffe41aa6d2998852a710c70a0,17.50,91.15,108.65,520.86
49031,8272b63d03f5f79c56e9e4120aec44ef,31.80,164.37,196.17,516.89
94113,f9ccaff7267fd0cf076e795b1fae8b69,3.00,15.23,18.23,507.67
22086,3aa41f0c241b0ef62147d849b02313be,4.90,24.84,29.74,506.94


#### Highest Freight-Burden Orders

The two highest freight-burden orders each contain R$0.85 in merchandise and R$18.23 in freight, producing a ratio of 2,144.71%. Their extreme ratios reflect the very small merchandise-value denominator.

Other high-burden orders also have freight charges substantially above merchandise value. For example, one order contains R$31.80 in merchandise and R$164.37 in freight, giving a burden of 516.89%.

These ratios are consistent with the recorded amounts, but the underlying reasons cannot be established from this comparison alone. The orders remain included, and high freight burden is treated as a descriptive commercial characteristic rather than a confirmed data error.

### 6.2 Assign Freight-Burden Bands

Orders are classified into six descriptive freight-burden bands: zero freight, above 0% to 10%, above 10% to 25%, above 25% to 50%, above 50% to 100%, and above 100%.

Zero-freight orders are identified separately. The remaining boundaries distinguish progressively larger freight charges relative to merchandise value. These reporting bands are analytical choices rather than established standards for acceptable shipping costs.

In [46]:
# Assign orders to freight-burden bands, keeping zero freight separate.
freight_burden = order_commercial_summary['freight_burden_pct']

freight_band_conditions = [
    freight_burden.eq(0),
    freight_burden.gt(0) & freight_burden.le(10),
    freight_burden.gt(10) & freight_burden.le(25),
    freight_burden.gt(25) & freight_burden.le(50),
    freight_burden.gt(50) & freight_burden.le(100),
    freight_burden.gt(100)
]

freight_band_labels = ['Zero freight','0-10%', '10-25%', '25-50%', '50-100%', 'Above 100%']

assigned_freight_bands = np.select(freight_band_conditions, freight_band_labels, default='unclassified')

# Set the freight-burden bands in ascending category order.
order_commercial_summary['freight_burden_band'] = pd.Categorical(assigned_freight_bands, categories=freight_band_labels, ordered=True)

In [47]:
# Validate band coverage and display order counts in band order.
print("Table shape:", order_commercial_summary.shape)

print("Unexpected band labels:", (assigned_freight_bands == 'unclassified').sum())

print("Missing freight-burden bands:", order_commercial_summary['freight_burden_band'].isna().sum())

print("Duplicate order IDs:", order_commercial_summary['order_id'].duplicated().sum())

print("Zero-freight band count:", order_commercial_summary['freight_burden_band'].eq('Zero freight').sum())

print("Above-100% band count:", order_commercial_summary['freight_burden_band'].eq('Above 100%').sum())

freight_burden_band_counts = (order_commercial_summary['freight_burden_band'].value_counts(sort=False))

print("Sum of band counts:", freight_burden_band_counts.sum())

print("Band counts reconcile:",freight_burden_band_counts.sum() == len(order_commercial_summary))

print("\nOrder counts by freight-burden band:")
display(freight_burden_band_counts)

Table shape: (96478, 7)
Unexpected band labels: 0
Missing freight-burden bands: 0
Duplicate order IDs: 0
Zero-freight band count: 336
Above-100% band count: 3075
Sum of band counts: 96478
Band counts reconcile: True

Order counts by freight-burden band:


freight_burden_band
Zero freight      336
0-10%           14925
10-25%          38168
25-50%          27778
50-100%         12196
Above 100%       3075
Name: count, dtype: int64

In [48]:
# Summarize order counts, merchandise GMV, and freight totals by freight-burden band.
freight_burden_summary = order_commercial_summary.groupby('freight_burden_band', as_index=False, observed=False).agg(order_count=('order_id', 'nunique'),
                                                                                                                     merchandise_gmv=('order_merchandise_value', 'sum'),
                                                                                                                     total_freight_value=('order_freight_value', 'sum'))


# Calculate each band's share of orders, merchandise GMV, and freight value.
total_delivered_orders = order_commercial_summary['order_id'].nunique()
total_merchandise_gmv = order_commercial_summary['order_merchandise_value'].sum()
total_freight_value = order_commercial_summary['order_freight_value'].sum()

freight_burden_summary['order_share_pct'] = (freight_burden_summary['order_count'] / total_delivered_orders) * 100


freight_burden_summary['gmv_share_pct'] = (freight_burden_summary['merchandise_gmv'] / total_merchandise_gmv) * 100

freight_burden_summary['freight_share_pct'] = (freight_burden_summary['total_freight_value'] / total_freight_value) * 100

In [49]:
# Validate summary completeness and reconcile counts, monetary totals, and percentage shares.
print("Summary shape:", freight_burden_summary.shape)

print("Duplicate bands:", freight_burden_summary['freight_burden_band'].duplicated().sum())

print("Missing values:", freight_burden_summary.isna().sum().sum())
print("Total order count:", freight_burden_summary['order_count'].sum())

print("Order counts reconcile:", freight_burden_summary['order_count'].sum() == total_delivered_orders)

print("Merchandise GMV reconciles:", abs(freight_burden_summary['merchandise_gmv'].sum() - total_merchandise_gmv) < 0.01)

print("Freight value reconciles:", abs(freight_burden_summary['total_freight_value'].sum() - total_freight_value) < 0.01)

for column in ['order_share_pct', 'gmv_share_pct', 'freight_share_pct']:
    print(
        f"{column} totals approximately 100:",
        abs(freight_burden_summary[column].sum() - 100) < 1e-8
    )

print("Zero-freight band total freight:", freight_burden_summary.loc[freight_burden_summary['freight_burden_band'].eq('Zero freight'),'total_freight_value'].sum())

# Display the freight-burden summary in band order, rounded only for presentation.
display(freight_burden_summary.sort_values('freight_burden_band').round(2))

Summary shape: (6, 7)
Duplicate bands: 0
Missing values: 0
Total order count: 96478
Order counts reconcile: True
Merchandise GMV reconciles: True
Freight value reconciles: True
order_share_pct totals approximately 100: True
gmv_share_pct totals approximately 100: True
freight_share_pct totals approximately 100: True
Zero-freight band total freight: 0.0


,freight_burden_band,order_count,merchandise_gmv,total_freight_value,order_share_pct,gmv_share_pct,freight_share_pct
0,Zero freight,336,37531.57,0.00,0.35,0.28,0.00
1,0-10%,14925,5572645.23,301859.40,15.47,42.15,13.73
2,10-25%,38168,5032721.48,811505.70,39.56,38.06,36.92
3,25-50%,27778,2034799.10,686316.65,28.79,15.39,31.22
4,50-100%,12196,478179.36,308591.50,12.64,3.62,14.04
5,Above 100%,3075,65621.37,90002.39,3.19,0.50,4.09


#### Freight-Burden Band Summary

The six freight-burden bands cover all 96,478 delivered orders. Order counts, merchandise GMV, and freight totals reconcile with the source, and all three percentage-share columns total approximately 100%.

The above-10%-to-25% band contains the largest share of orders at 39.56%. Orders with positive freight burden up to 10% represent 15.47% of orders but contribute 42.15% of merchandise GMV.

For 3,075 orders (3.19%), freight exceeds merchandise value. These orders contribute 0.50% of merchandise GMV and 4.09% of freight value. The 336 zero-freight orders remain included and correctly contribute no freight value.

These comparisons describe the distribution of recorded freight charges relative to merchandise value. Because merchandise value is the denominator of freight burden, larger baskets can have lower ratios even when their freight charges are substantial. The results do not establish shipping profitability or the effect of freight charges on purchase decisions.

## 7. Identify Commercial Segments

Order merchandise-value bands and freight-burden bands are combined to identify groups with substantial sales contribution or high freight charges relative to merchandise value.

These segments describe delivered orders rather than individual customers. They support further investigation without assuming profitability or explaining why customers completed their purchases.

### 7.1 Summarize Order Value and Freight Burden Together

Each observed combination is summarized using order count, merchandise GMV, and total freight value. This reveals how freight burden varies across order-value bands while preserving the complete delivered-order population.

In [50]:
# Summarize observed combinations of order-value and freight-burden bands.
commercial_segment_summary = order_commercial_summary.groupby(['order_value_band', 'freight_burden_band'], as_index=False, observed=True).agg(order_count=('order_id', 'nunique'),merchandise_gmv=('order_merchandise_value', 'sum'),total_freight_value=('order_freight_value', 'sum'))

# Validate segment uniqueness and reconcile order counts and monetary totals.
print("Summary shape:", commercial_segment_summary.shape)

print("Duplicate band combinations:", commercial_segment_summary[['order_value_band', 'freight_burden_band']].duplicated().sum())

print("Missing values:", commercial_segment_summary.isna().sum().sum())

print("Every segment has at least one order:", commercial_segment_summary['order_count'].gt(0).all())

print("Total order count:", commercial_segment_summary['order_count'].sum())

print("Order counts reconcile:", commercial_segment_summary['order_count'].sum() == order_commercial_summary['order_id'].nunique())

print("Merchandise GMV reconciles:", abs(commercial_segment_summary['merchandise_gmv'].sum() - order_commercial_summary['order_merchandise_value'].sum()) < 0.01)

print("Freight totals reconcile:", abs(commercial_segment_summary['total_freight_value'].sum() - order_commercial_summary['order_freight_value'].sum()) < 0.01)

# Display commercial segments ordered by merchandise GMV.
display(commercial_segment_summary.sort_values('merchandise_gmv', ascending=False).reset_index(drop=True).round(2))

Summary shape: (33, 5)
Duplicate band combinations: 0
Missing values: 0
Every segment has at least one order: True
Total order count: 96478
Order counts reconcile: True
Merchandise GMV reconciles: True
Freight totals reconcile: True


,order_value_band,freight_burden_band,order_count,merchandise_gmv,total_freight_value
0,150-400,0-10%,8679,2166289.99,148380.44
1,400-1000,0-10%,2806,1758930.22,82535.10
2,150-400,10-25%,7905,1713663.62,261577.84
3,100-150,10-25%,11122,1402281.87,217678.23
4,1000+,0-10%,807,1329625.91,44561.43
5,50-100,10-25%,15398,1220966.02,223744.02
6,50-100,25-50%,9723,678046.97,222204.32
7,0-50,25-50%,13194,492580.57,178985.13
8,400-1000,10-25%,777,458975.27,67538.52
9,150-400,25-50%,1806,395101.62,129482.55


In [51]:
# Calculate each commercial segment's share of orders, merchandise GMV, and freight value.
commercial_segment_summary['order_share_pct'] = (commercial_segment_summary['order_count']/ order_commercial_summary['order_id'].nunique() * 100)

commercial_segment_summary['gmv_share_pct'] = (commercial_segment_summary['merchandise_gmv']/ order_commercial_summary['order_merchandise_value'].sum() * 100)

commercial_segment_summary['freight_share_pct'] = (commercial_segment_summary['total_freight_value']/ order_commercial_summary['order_freight_value'].sum() * 100)

In [52]:
# Validate percentage completeness and confirm each share column totals approximately 100%.
segment_share_columns = ['order_share_pct', 'gmv_share_pct', 'freight_share_pct']

print("Summary shape:", commercial_segment_summary.shape)

print("Missing percentage values:", commercial_segment_summary[segment_share_columns].isna().sum().sum())

for column in segment_share_columns:
    print(f"{column} totals approximately 100:", abs(commercial_segment_summary[column].sum() - 100) < 1e-8)
    print(f"{column} values are between 0 and 100:", commercial_segment_summary[column].between(0, 100).all())

# Display the ten largest commercial segments by merchandise GMV.
display(commercial_segment_summary.sort_values('merchandise_gmv', ascending=False).head(10).reset_index(drop=True).round(2))

Summary shape: (33, 8)
Missing percentage values: 0
order_share_pct totals approximately 100: True
order_share_pct values are between 0 and 100: True
gmv_share_pct totals approximately 100: True
gmv_share_pct values are between 0 and 100: True
freight_share_pct totals approximately 100: True
freight_share_pct values are between 0 and 100: True


,order_value_band,freight_burden_band,order_count,merchandise_gmv,total_freight_value,order_share_pct,gmv_share_pct,freight_share_pct
0,150-400,0-10%,8679,2166289.99,148380.44,9.00,16.38,6.75
1,400-1000,0-10%,2806,1758930.22,82535.10,2.91,13.30,3.75
2,150-400,10-25%,7905,1713663.62,261577.84,8.19,12.96,11.90
3,100-150,10-25%,11122,1402281.87,217678.23,11.53,10.61,9.90
4,1000+,0-10%,807,1329625.91,44561.43,0.84,10.06,2.03
5,50-100,10-25%,15398,1220966.02,223744.02,15.96,9.23,10.18
6,50-100,25-50%,9723,678046.97,222204.32,10.08,5.13,10.11
7,0-50,25-50%,13194,492580.57,178985.13,13.68,3.73,8.14
8,400-1000,10-25%,777,458975.27,67538.52,0.81,3.47,3.07
9,150-400,25-50%,1806,395101.62,129482.55,1.87,2.99,5.89


#### Commercial Segment Contribution Summary

The analysis identifies 33 observed combinations of order merchandise value and freight burden. These segments cover all 96,478 delivered orders, and their merchandise GMV and freight totals reconcile with the source.

The largest GMV segment contains orders above R$150 and up to R$400 with positive freight burden up to 10%. Its 8,679 orders represent 9.00% of delivered orders and contribute 16.38% of merchandise GMV.

Orders above R$1,000 with positive freight burden up to 10% form a smaller but substantial-value segment: 807 orders represent 0.84% of orders and contribute 10.06% of merchandise GMV.

Among orders whose freight exceeds merchandise value, 2,919 of 3,075 have merchandise values up to R$50. This identifies low-value orders with high relative freight charges as a group for further investigation.

These segments describe order-level commercial patterns. High merchandise contribution does not establish profitability, and high freight burden does not demonstrate that freight caused dissatisfaction or discouraged purchases.

In [53]:
# Inspect available columns to select order-level delivery and review measures.
print(delivered_order_scope.columns.tolist())

['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'flag_future_shipping_limit_date', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'flag_delivered_missing_customer_date', 'flag_delivered_missing_carrier_date', 'flag_delivered_missing_approval_date', 'flag_delivered_missing_critical_timestamp', 'critical_timestamp_flag_count', 'flag_carrier_before_purchase', 'flag_customer_before_carrier', 'flag_carrier_before_approval', 'flag_lifecycle_timestamp_exception', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'flag_missing_catalog_attributes', 'flag_missing_physical_attributes', 'flag_zero_p

## 8. Compare Payment Methods Using Commercial and Customer-Experience Measures

Orders are compared by dominant payment method using merchandise value, freight burden, review scores, and delivery performance. Commercial measures come from the validated order-level commercial summary, while review and delivery measures come from the delivered-order scope.

Review and delivery coverage are inspected before calculating averages or rates. Missing observations will not be replaced with zero, and each measure will use an explicitly defined eligible population. These comparisons describe associations rather than effects caused by payment method.

### 8.1 Inspect Review and Delivery Measure Coverage

Review availability, delivery-status labels, and delivery-quality flags are examined to determine which orders can contribute to each customer-experience measure.

In [54]:
# Inspect review-score availability and missing-review indicators.
print("Missing average review scores:", delivered_order_scope['average_review_score'].isna().sum())

print("\nMissing-review flag counts:\n", delivered_order_scope['flag_missing_review_record'].value_counts(dropna=False))

print("\nMinimum average review score:", delivered_order_scope['average_review_score'].min())

print("Maximum average review score:", delivered_order_scope['average_review_score'].max())

print("All available average review scores are between 1 and 5:", delivered_order_scope['average_review_score'].dropna().between(1, 5).all())

Missing average review scores: 646

Missing-review flag counts:
 flag_missing_review_record
False    95832
True       646
Name: count, dtype: int64

Minimum average review score: 1.0
Maximum average review score: 5.0
All available average review scores are between 1 and 5: True


In [55]:
# Inspect delivery-status labels and missing delivery durations.
print("\nDelivery-status counts:\n", delivered_order_scope['delivery_status'].value_counts(dropna=False))

print("\nMissing delivery durations:", delivered_order_scope['delivery_duration_days'].isna().sum())

print("Negative delivery durations:", delivered_order_scope['delivery_duration_days'].lt(0).sum())



Delivery-status counts:
 delivery_status
early      88644
late        6534
on_time     1292
missing        8
Name: count, dtype: int64

Missing delivery durations: 8
Negative delivery durations: 0


In [56]:
# Inspect delivery-quality flags before defining eligible comparison populations.
for column in ['flag_delivered_missing_critical_timestamp', 'flag_lifecycle_timestamp_exception']:
    print(f"\n{column} counts:\n", delivered_order_scope[column].value_counts(dropna=False))


flag_delivered_missing_critical_timestamp counts:
 flag_delivered_missing_critical_timestamp
False    96455
True        23
Name: count, dtype: int64

flag_lifecycle_timestamp_exception counts:
 flag_lifecycle_timestamp_exception
False    95105
True      1373
Name: count, dtype: int64


#### Review and Delivery Coverage Summary

Review scores are available for 95,832 of the 96,478 delivered orders. The remaining 646 orders are flagged as missing review records. All available average review scores fall between 1 and 5.

Delivery status identifies 88,644 early orders, 6,534 late orders, 1,292 on-time orders, and eight orders with missing status information. Delivery duration is missing for eight orders, with no negative durations observed.

Broader quality flags identify 23 orders with missing critical timestamps and 1,373 orders with lifecycle timestamp exceptions. These flags cover multiple stages of the order lifecycle, so eligibility will be determined using the timestamps required for each metric rather than excluding every flagged order.

These coverage counts describe the full delivered-order population. Coverage by payment method will be calculated after joining the relevant measures.

In [57]:
# Inspect missing timestamps required for delivery-duration and timeliness comparisons.
delivery_timestamp_columns = ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']

print("Missing delivery-related timestamps:\n", delivered_order_scope[delivery_timestamp_columns].isna().sum())

# Confirm that review-score availability agrees with the missing-review flag.
print("\nReview-score availability matches the missing-review flag:", (delivered_order_scope['average_review_score'].isna() == delivered_order_scope['flag_missing_review_record']).all())

# Inspect missing calendar delivery differences and their delivery-status labels.
missing_calendar_difference_mask = (delivered_order_scope['delivery_calendar_difference_days'].isna())

print("\nMissing calendar delivery differences:",missing_calendar_difference_mask.sum())

display(delivered_order_scope.loc[missing_calendar_difference_mask,['order_id','delivery_status','order_delivered_customer_date','order_estimated_delivery_date']])

Missing delivery-related timestamps:
 order_purchase_timestamp         0
order_delivered_customer_date    8
order_estimated_delivery_date    0
dtype: int64

Review-score availability matches the missing-review flag: True

Missing calendar delivery differences: 8


,order_id,delivery_status,order_delivered_customer_date,order_estimated_delivery_date
5723,0d3268bad9b086af767785e3f0fc0133,missing,NaT,2018-07-24
14141,20edc82cf5400ce95e1afacc25798b31,missing,NaT,2018-07-19
19205,2d1e2d5bf4dc7227b3bfebb81328c15f,missing,NaT,2017-12-18
19398,2d858f451373b04fb5c984a1cc2defaf,missing,NaT,2017-06-23
19933,2ebdfc4f15f23b91474edf87475f108e,missing,NaT,2018-07-30
73662,ab7c89dc1bf4a1ead9d6ec1ec8968a84,missing,NaT,2018-06-26
99440,e69f75a717d64fc5ecdfae42b2e8e086,missing,NaT,2018-07-30
105841,f5dd62b788049ad9fc0526e3ad11a097,missing,NaT,2018-07-16


#### Metric-Specific Eligibility Decision

Purchase and estimated delivery timestamps are complete. Eight orders lack actual customer delivery timestamps; these orders also have missing calendar delivery differences and a delivery status of missing.

Review comparisons will include orders with available average review scores between 1 and 5. Delivery-duration comparisons will require purchase and actual delivery timestamps, together with an available non-negative duration. Late-delivery rates will require actual and estimated delivery timestamps, an available calendar difference, and a recognized delivery status.

Each metric will use its own eligible-order denominator. Missing delivery information will not be interpreted as an on-time delivery, and broader lifecycle flags will remain available for context.

### 8.2 Prepare Order-Level Customer-Experience Measures

A separate table retains one row per delivered order with review and delivery measures. Eligibility flags identify which orders can contribute to each comparison before these measures are joined with commercial and payment information.

In [58]:
# Select order-level review and delivery measures without carrying item-level attributes.
order_experience_summary = delivered_order_scope[['order_id', 'average_review_score', 'delivery_duration_days', 'delivery_calendar_difference_days', 'delivery_status']].copy()

# Define separate eligibility flags for review, delivery-duration, and delivery-status comparisons.
order_experience_summary['flag_review_eligible'] = delivered_order_scope['average_review_score'].between(1, 5).fillna(False)

order_experience_summary['flag_delivery_duration_eligible'] = (
    delivered_order_scope['order_purchase_timestamp'].notna()
    & delivered_order_scope['order_delivered_customer_date'].notna()
    & delivered_order_scope['delivery_duration_days'].notna()
    & delivered_order_scope['delivery_duration_days'].ge(0)
).fillna(False)

order_experience_summary['flag_delivery_status_eligible'] = (
    delivered_order_scope['order_delivered_customer_date'].notna()
    & delivered_order_scope['order_estimated_delivery_date'].notna()
    & delivered_order_scope['delivery_calendar_difference_days'].notna()
    & delivered_order_scope['delivery_status'].isin(['early', 'on_time', 'late']))

In [59]:
# Validate order uniqueness, eligibility coverage, and completeness of the flags.
eligibility_columns = ['flag_review_eligible', 'flag_delivery_duration_eligible', 'flag_delivery_status_eligible']

print("Table shape:", order_experience_summary.shape)

print("Duplicate order IDs:", order_experience_summary['order_id'].duplicated().sum())

print("\nMissing eligibility flags:\n", order_experience_summary[eligibility_columns].isna().sum())

print("\nEligible-order counts:\n", order_experience_summary[eligibility_columns].sum())

print("\nLate orders among status-eligible orders:", (order_experience_summary['flag_delivery_status_eligible'] & order_experience_summary['delivery_status'].eq('late')).sum())

Table shape: (96478, 8)
Duplicate order IDs: 0

Missing eligibility flags:
 flag_review_eligible               0
flag_delivery_duration_eligible    0
flag_delivery_status_eligible      0
dtype: int64

Eligible-order counts:
 flag_review_eligible               95832
flag_delivery_duration_eligible    96470
flag_delivery_status_eligible      96470
dtype: int64

Late orders among status-eligible orders: 6534


#### Order-Level Customer-Experience Validation Summary

The customer-experience table contains 96,478 unique delivered orders and eight columns, with no duplicate order IDs or missing eligibility flags.

Review comparisons have 95,832 eligible orders. Delivery-duration and delivery-status comparisons each have 96,470 eligible orders, including 6,534 late deliveries. Original missing metric values remain unchanged, and each comparison will use its own eligible-order denominator.

### 8.3 Combine Commercial, Payment, and Customer-Experience Measures

The validated commercial summary is joined with payment classifications and customer-experience measures at order grain. One-to-one left joins preserve all delivered orders, including the order without payment information.

The resulting table retains merchandise-value bands, freight-burden bands, and metric-specific eligibility flags. Payment-method comparisons will use orders with available payment classifications, with review and delivery denominators calculated separately within each group.

In [60]:
# Combine commercial measures with order-level payment information, preserving all delivered orders.
order_commercial_payment_experience = pd.merge(order_commercial_summary, order_payment_analysis, on='order_id', how='left', validate='one_to_one', indicator='payment_coverage')

# Add customer-experience measures and eligibility flags using a one-to-one join.
order_commercial_payment_experience = pd.merge(order_commercial_payment_experience, order_experience_summary, on='order_id', how='left', validate='one_to_one', indicator='experience_coverage')

In [61]:
# Validate join coverage, order uniqueness, and preservation of commercial totals.
print("Table shape:", order_commercial_payment_experience.shape)

print("Duplicate order IDs:", order_commercial_payment_experience['order_id'].duplicated().sum())

print("\nPayment coverage:\n", order_commercial_payment_experience['payment_coverage'].value_counts())

print("\nExperience coverage:\n", order_commercial_payment_experience['experience_coverage'].value_counts())

print("\nMissing dominant payment types:", order_commercial_payment_experience['dominant_payment_type'].isna().sum())

Table shape: (96478, 27)
Duplicate order IDs: 0

Payment coverage:
 payment_coverage
both          96477
left_only         1
right_only        0
Name: count, dtype: int64

Experience coverage:
 experience_coverage
both          96478
left_only         0
right_only        0
Name: count, dtype: int64

Missing dominant payment types: 1


In [62]:
eligibility_columns = ['flag_review_eligible', 'flag_delivery_duration_eligible', 'flag_delivery_status_eligible']

print("\nMissing eligibility flags:\n", order_commercial_payment_experience[eligibility_columns].isna().sum())

print("\nMerchandise totals reconcile:", abs(order_commercial_payment_experience['order_merchandise_value'].sum() - order_commercial_summary['order_merchandise_value'].sum()) < 0.01)

print("Freight totals reconcile:", abs(order_commercial_payment_experience['order_freight_value'].sum() - order_commercial_summary['order_freight_value'].sum()) < 0.01)

print("\nEligible-order counts:\n", order_commercial_payment_experience[eligibility_columns].sum())


Missing eligibility flags:
 flag_review_eligible               0
flag_delivery_duration_eligible    0
flag_delivery_status_eligible      0
dtype: int64

Merchandise totals reconcile: True
Freight totals reconcile: True

Eligible-order counts:
 flag_review_eligible               95832
flag_delivery_duration_eligible    96470
flag_delivery_status_eligible      96470
dtype: int64


#### Combined Analysis Table Validation Summary

The combined table contains 96,478 unique delivered orders and 27 columns. Payment information is available for 96,477 orders, while customer-experience table coverage includes all 96,478 orders.

One order remains without a dominant payment classification. Eligibility flags are complete, with 95,832 review-eligible orders and 96,470 orders eligible for each delivery comparison. Merchandise and freight totals reconcile with the commercial summary.

Complete join coverage does not imply complete review or delivery observations. Metric-specific eligibility flags remain necessary when calculating averages and rates.

### 8.4 Compare Commercial and Customer-Experience Measures by Payment Method

Orders with available payment information are grouped by dominant payment method. Comparisons include merchandise value, freight burden, average order review score, delivery duration, and late-delivery rate.

Review and delivery measures use their respective eligible-order populations within each payment-method group. Counts of eligible orders are retained alongside the results to make denominators explicit. Differences between payment-method groups are descriptive and may reflect differences in order or customer characteristics.

In [63]:
# Select payment-covered orders for dominant-payment-method comparisons.
payment_method_comparison_scope = order_commercial_payment_experience.loc[order_commercial_payment_experience['payment_coverage'].eq('both')].copy()

# Prepare eligible review scores, delivery durations, and late-delivery indicators.
payment_method_comparison_scope['eligible_review_score'] = payment_method_comparison_scope['average_review_score'].where(payment_method_comparison_scope['flag_review_eligible'])

payment_method_comparison_scope['eligible_delivery_duration_days'] = payment_method_comparison_scope['delivery_duration_days'].where(payment_method_comparison_scope['flag_delivery_duration_eligible'])

payment_method_comparison_scope['eligible_late_indicator'] = payment_method_comparison_scope['delivery_status'].eq('late').astype(float).where(payment_method_comparison_scope['flag_delivery_status_eligible'])

In [64]:
# Validate comparison coverage and metric-specific denominators.
print("Comparison scope shape:", payment_method_comparison_scope.shape)

print("Duplicate order IDs:", payment_method_comparison_scope['order_id'].duplicated().sum())

print("Missing dominant payment types:", payment_method_comparison_scope['dominant_payment_type'].isna().sum())

metric_eligibility_pairs = [
    ('eligible_review_score', 'flag_review_eligible'),
    ('eligible_delivery_duration_days', 'flag_delivery_duration_eligible'),
    ('eligible_late_indicator', 'flag_delivery_status_eligible')
]

for metric, flag in metric_eligibility_pairs:
    print(f"\n{metric} non-missing count:", payment_method_comparison_scope[metric].notna().sum())
    print(
        f"{metric} availability matches eligibility for every order:", (payment_method_comparison_scope[metric].notna() == payment_method_comparison_scope[flag]).all())

print("\nAvailable late indicators contain only 0 and 1:",payment_method_comparison_scope['eligible_late_indicator'].dropna().isin([0.0, 1.0]).all())

Comparison scope shape: (96477, 30)
Duplicate order IDs: 0
Missing dominant payment types: 0

eligible_review_score non-missing count: 95831
eligible_review_score availability matches eligibility for every order: True

eligible_delivery_duration_days non-missing count: 96469
eligible_delivery_duration_days availability matches eligibility for every order: True

eligible_late_indicator non-missing count: 96469
eligible_late_indicator availability matches eligibility for every order: True

Available late indicators contain only 0 and 1: True


In [65]:
# Compare commercial values and eligible customer-experience measures by dominant payment method.
payment_method_experience_summary = payment_method_comparison_scope.groupby('dominant_payment_type', as_index=False).agg(order_count=('order_id', 'nunique'),
                                                                                                                         mean_merchandise_value=('order_merchandise_value', 'mean'),
                                                                                                                         median_merchandise_value=('order_merchandise_value', 'median'),
                                                                                                                         median_freight_burden_pct=('freight_burden_pct', 'median'),
                                                                                                                         review_eligible_order_count=('eligible_review_score', 'count'),
                                                                                                                         mean_order_review_score=('eligible_review_score', 'mean'),
                                                                                                                         delivery_duration_eligible_order_count=('eligible_delivery_duration_days', 'count'),
                                                                                                                         mean_delivery_duration_days=('eligible_delivery_duration_days', 'mean'),
                                                                                                                         delivery_status_eligible_order_count=('eligible_late_indicator', 'count'),
                                                                                                                         late_order_count=('eligible_late_indicator', 'sum'))

# Calculate late-delivery rates using status-eligible orders within each payment-method group.
payment_method_experience_summary['late_delivery_rate_pct'] = (payment_method_experience_summary['late_order_count']/ payment_method_experience_summary['delivery_status_eligible_order_count'].replace(0, np.nan)* 100)



In [66]:
# Validate order coverage and metric-specific denominators.
print("Summary shape:", payment_method_experience_summary.shape)

print("Duplicate payment methods:", payment_method_experience_summary['dominant_payment_type'].duplicated().sum())

print("Order-count sum:", payment_method_experience_summary['order_count'].sum())

print("Order counts reconcile:", payment_method_experience_summary['order_count'].sum() == len(payment_method_comparison_scope))

denominator_checks = [
    ('review_eligible_order_count', 'eligible_review_score'),
    ('delivery_duration_eligible_order_count', 'eligible_delivery_duration_days'),
    ('delivery_status_eligible_order_count', 'eligible_late_indicator')
]

for count_column, source_column in denominator_checks:
    print(f"\n{count_column} sum:", payment_method_experience_summary[count_column].sum())
    print(f"{count_column} reconciles:", payment_method_experience_summary[count_column].sum() == payment_method_comparison_scope[source_column].notna().sum())
    print(f"{count_column} does not exceed group order count:", payment_method_experience_summary[count_column].le(payment_method_experience_summary['order_count']).all())

print("\nLate-order counts reconcile:", payment_method_experience_summary['late_order_count'].sum() == payment_method_comparison_scope['eligible_late_indicator'].sum())

print("Late-order counts do not exceed eligible orders:", payment_method_experience_summary['late_order_count'].le(payment_method_experience_summary['delivery_status_eligible_order_count']).all())

print("Mean review scores are between 1 and 5:", payment_method_experience_summary['mean_order_review_score'].between(1, 5).all())

print("Late-delivery rates are between 0 and 100:", payment_method_experience_summary['late_delivery_rate_pct'].between(0, 100).all())

# Display the comparison rounded only for presentation.
display(payment_method_experience_summary.sort_values('order_count', ascending=False).reset_index(drop=True).round(2))

Summary shape: (4, 12)
Duplicate payment methods: 0
Order-count sum: 96477
Order counts reconcile: True

review_eligible_order_count sum: 95831
review_eligible_order_count reconciles: True
review_eligible_order_count does not exceed group order count: True

delivery_duration_eligible_order_count sum: 96469
delivery_duration_eligible_order_count reconciles: True
delivery_duration_eligible_order_count does not exceed group order count: True

delivery_status_eligible_order_count sum: 96469
delivery_status_eligible_order_count reconciles: True
delivery_status_eligible_order_count does not exceed group order count: True

Late-order counts reconcile: True
Late-order counts do not exceed eligible orders: True
Mean review scores are between 1 and 5: True
Late-delivery rates are between 0 and 100: True


,dominant_payment_type,order_count,mean_merchandise_value,median_merchandise_value,median_freight_burden_pct,review_eligible_order_count,mean_order_review_score,delivery_duration_eligible_order_count,mean_delivery_duration_days,delivery_status_eligible_order_count,late_order_count,late_delivery_rate_pct
0,credit_card,72785,143.24,89.90,21.61,72298,4.16,72778,12.35,72778,4869.0,6.69
1,boleto,19191,121.40,74.90,25.28,19062,4.16,19191,13.49,19191,1405.0,7.32
2,voucher,3017,95.57,59.99,27.40,2993,4.13,3017,12.50,3017,180.0,5.97
3,debit_card,1484,119.86,70.45,24.91,1478,4.24,1483,10.79,1483,79.0,5.33


#### Payment-Method Commercial and Customer-Experience Summary

The comparison covers 96,477 payment-classified delivered orders across four dominant payment methods. Review averages use 95,831 eligible orders, while delivery measures use 96,469 eligible orders. All group counts and metric-specific denominators reconcile with the comparison scope.

Credit-card-dominant orders have the highest mean merchandise value of R$143.24 and median of R$89.90. Voucher-dominant orders have the lowest mean and median merchandise values, at R$95.57 and R$59.99, and the highest median freight burden at 27.40%.

Average order review scores range from 4.13 for voucher-dominant orders to 4.24 for debit-card-dominant orders. Credit card and boleto both have average scores of 4.16 when rounded to two decimal places. Each eligible order receives equal weight in these review averages.

Boleto-dominant orders have the longest average delivery duration at 13.49 days and the highest late-delivery rate at 7.32%. Debit-card-dominant orders have the shortest average duration at 10.79 days and the lowest late-delivery rate at 5.33%.

These are descriptive comparisons across groups of different sizes and potentially different product, regional, and customer compositions. They do not establish that payment method causes differences in merchandise value, reviews, or delivery performance.

## 9. Final Validation and Export

Final analytical tables are checked for their intended grain, population coverage, and reconciliation of key totals before export. Exported files will be read back to confirm that row counts, identifiers, and important values are preserved.

Orders with missing payment information and identified discrepancies remain explicitly represented in the relevant order-level outputs.

In [67]:
# Confirm the shape and order-ID uniqueness of the final order-level analytical tables.
final_order_tables = {'order_commercial_summary': order_commercial_summary,
                      'order_payment_reconciliation': order_payment_reconciliation,
                      'order_commercial_payment_experience': order_commercial_payment_experience
                      }

expected_shapes = {'order_commercial_summary': (96478, 7),
                   'order_payment_reconciliation': (96478, 19),
                   'order_commercial_payment_experience': (96478, 27)
                   }

expected_order_ids = set(delivered_order_scope['order_id'])

for table_name, table in final_order_tables.items():
    print(f"\n{table_name}")
    print("Shape:", table.shape)
    print("Shape matches expected:", table.shape == expected_shapes[table_name])
    print("Duplicate order IDs:", table['order_id'].duplicated().sum())
    print("Missing order IDs:", table['order_id'].isna().sum())

# Confirm that every table contains exactly the validated delivered-order population.
print("Order-ID coverage matches delivered-order scope:", set(table['order_id']) == expected_order_ids)


order_commercial_summary
Shape: (96478, 7)
Shape matches expected: True
Duplicate order IDs: 0
Missing order IDs: 0

order_payment_reconciliation
Shape: (96478, 19)
Shape matches expected: True
Duplicate order IDs: 0
Missing order IDs: 0

order_commercial_payment_experience
Shape: (96478, 27)
Shape matches expected: True
Duplicate order IDs: 0
Missing order IDs: 0
Order-ID coverage matches delivered-order scope: True


In [68]:
# Confirm that each final table contains exactly the validated delivered-order population.
expected_order_ids = set(delivered_order_scope['order_id'])

for table_name, table in final_order_tables.items():
    print(f"{table_name} — order-ID coverage matches:", set(table['order_id']) == expected_order_ids)

order_commercial_summary — order-ID coverage matches: True
order_payment_reconciliation — order-ID coverage matches: True
order_commercial_payment_experience — order-ID coverage matches: True


#### Final Order-Level Population Validation

The commercial summary, payment reconciliation table, and combined commercial-payment-experience table each contain exactly 96,478 delivered orders. Their shapes match expectations, with no duplicate or missing order IDs.

The order-ID population in each table exactly matches the validated delivered-order scope. The order without payment information remains included in all three outputs.

### 9.1 Export Analytical Tables

Order-level outputs are exported as Parquet files to preserve data types and missing values. Summary tables are exported as CSV files for reporting and inspection.

Numeric values retain their stored precision during export. Files will be read back and compared with their in-memory source tables before export validation is considered complete.

In [69]:
# Create the output folder for payment and commercial analytical tables.
payment_commercial_output_path = (processed_data_path / "payment_and_commercial_analysis")
payment_commercial_output_path.mkdir(parents=True, exist_ok=True)

# Export the three order-level tables as Parquet without the DataFrame index.
order_level_export_tables = {
    'order_commercial_summary': order_commercial_summary,
    'order_payment_reconciliation': order_payment_reconciliation,
    'order_commercial_payment_experience': order_commercial_payment_experience
}

for table_name, table in order_level_export_tables.items():
    output_file = payment_commercial_output_path / f"{table_name}.parquet"
    table.to_parquet(output_file, index=False)
    print(f"Exported: {output_file.name} — {table.shape}")

# Export the ten summary tables as CSV without rounding or the DataFrame index.
summary_export_tables = {
    'dominant_payment_method_summary': dominant_payment_method_summary,
    'dominant_payment_value_summary': dominant_payment_value_summary,
    'installment_group_summary': installment_group_summary,
    'payment_discrepancy_summary': payment_discrepancy_summary,
    'multiple_payment_discrepancy_summary': multiple_payment_discrepancy_summary,
    'quality_exception_discrepancy_summary': quality_exception_discrepancy_summary,
    'order_value_band_summary': order_value_band_summary,
    'freight_burden_summary': freight_burden_summary,
    'commercial_segment_summary': commercial_segment_summary,
    'payment_method_experience_summary': payment_method_experience_summary
    }

for table_name, table in summary_export_tables.items():
    output_file = payment_commercial_output_path / f"{table_name}.csv"
    table.to_csv(output_file, index=False)
    print(f"Exported: {output_file.name} — {table.shape}")

print("\nOutput folder:", payment_commercial_output_path.resolve())
print("Total tables exported:",len(order_level_export_tables) + len(summary_export_tables))

Exported: order_commercial_summary.parquet — (96478, 7)
Exported: order_payment_reconciliation.parquet — (96478, 19)
Exported: order_commercial_payment_experience.parquet — (96478, 27)
Exported: dominant_payment_method_summary.csv — (4, 3)
Exported: dominant_payment_value_summary.csv — (4, 4)
Exported: installment_group_summary.csv — (5, 5)
Exported: payment_discrepancy_summary.csv — (2, 5)
Exported: multiple_payment_discrepancy_summary.csv — (2, 4)
Exported: quality_exception_discrepancy_summary.csv — (2, 4)
Exported: order_value_band_summary.csv — (6, 5)
Exported: freight_burden_summary.csv — (6, 7)
Exported: commercial_segment_summary.csv — (33, 8)
Exported: payment_method_experience_summary.csv — (4, 12)

Output folder: C:\Users\Hp\DataAnalyst\data\processed\payment_and_commercial_analysis
Total tables exported: 13


### 9.2 Read Back and Validate Exported Files

All exported files are read back and compared with their in-memory source tables. Validation checks shape, column order, missing values, and data values.

Parquet outputs are checked for preserved data types. CSV comparisons allow for categorical and other type metadata that CSV does not store, while still checking labels, counts, and numeric values. Small floating-point differences are tolerated without rounding the source data.

In [70]:
validation_results = []
expected_order_ids = set(delivered_order_scope['order_id'])

# Read back the three Parquet outputs and compare them with their source tables.
for table_name, source_table in order_level_export_tables.items():
    file_path = payment_commercial_output_path / f"{table_name}.parquet"
    restored_table = pd.read_parquet(file_path)
    expected_table = source_table.reset_index(drop=True)

    # Check shape, column order, data types, missing values, and data values.
    pd.testing.assert_frame_equal(
        restored_table,
        expected_table,
        check_dtype=True,
        check_exact=True
    )

    # Confirm order-ID uniqueness and complete delivered-order coverage.
    assert restored_table['order_id'].notna().all(), f"{table_name}: missing order IDs"
    assert restored_table['order_id'].is_unique, f"{table_name}: duplicate order IDs"
    assert set(restored_table['order_id']) == expected_order_ids, (
        f"{table_name}: order-ID coverage mismatch"
    )

    validation_results.append({
        'file_name': file_path.name,
        'row_count': len(restored_table),
        'column_count': len(restored_table.columns),
        'validation_status': 'Passed'
    })

# Read back the ten CSV summaries and compare their values with the source tables.
for table_name, source_table in summary_export_tables.items():
    file_path = payment_commercial_output_path / f"{table_name}.csv"
    restored_table = pd.read_csv(file_path, float_precision='round_trip')
    expected_table = source_table.reset_index(drop=True).copy()

    # CSV preserves category labels but does not store categorical metadata.
    for column in expected_table.columns:
        if isinstance(expected_table[column].dtype, pd.CategoricalDtype):
            expected_table[column] = expected_table[column].astype(object)

    # Check missing-value locations explicitly.
    pd.testing.assert_frame_equal(
        restored_table.isna(),
        expected_table.isna()
    )

    # Compare labels and numeric values while allowing CSV type inference.
    pd.testing.assert_frame_equal(
        restored_table,
        expected_table,
        check_dtype=False,
        check_categorical=False,
        check_exact=False,
        rtol=1e-12,
        atol=1e-8
    )

    # Require exact preservation of count columns.
    for column in expected_table.columns:
        if column.endswith('_count'):
            pd.testing.assert_series_equal(
                restored_table[column],
                expected_table[column],
                check_dtype=False,
                check_exact=True
            )

    validation_results.append({
        'file_name': file_path.name,
        'row_count': len(restored_table),
        'column_count': len(restored_table.columns),
        'validation_status': 'Passed'
    })

# Report validation results for all 13 exported files.
export_validation_summary = pd.DataFrame(validation_results)

assert len(export_validation_summary) == 13, "Expected 13 validated files."

display(export_validation_summary)
print("Total files validated:", len(export_validation_summary))
print("All 13 exported files passed read-back validation.")

,file_name,row_count,column_count,validation_status
0,order_commercial_summary.parquet,96478,7,Passed
1,order_payment_reconciliation.parquet,96478,19,Passed
2,order_commercial_payment_experience.parquet,96478,27,Passed
3,dominant_payment_method_summary.csv,4,3,Passed
4,dominant_payment_value_summary.csv,4,4,Passed
5,installment_group_summary.csv,5,5,Passed
6,payment_discrepancy_summary.csv,2,5,Passed
7,multiple_payment_discrepancy_summary.csv,2,4,Passed
8,quality_exception_discrepancy_summary.csv,2,4,Passed
9,order_value_band_summary.csv,6,5,Passed


Total files validated: 13
All 13 exported files passed read-back validation.


#### Export and Read-Back Validation Summary

All 13 analytical outputs were exported and successfully validated after reading them back: three order-level Parquet files and ten summary CSV files.

Read-back comparisons confirmed preserved shapes, column order, missing-value locations, and data values within the specified numeric tolerance. Parquet data types were preserved, and CSV count columns matched exactly.

Each order-level output retains the complete population of 96,478 delivered orders, with unique, non-missing order IDs. The known missing-payment order and reconciliation discrepancies remain represented in the relevant outputs.

## 10. Key Findings and Limitations

Credit card is the dominant payment method for 75.44% of payment-covered delivered orders. Credit-card-dominant orders also have the highest mean merchandise value, at R$143.24.

Higher installment bands are associated with higher mean and median payment values. However, the 13–24 installment group contains only 179 payment records, and the comparison does not establish that installments cause higher spending.

Payment reconciliation identifies 95,908 matched orders, 569 orders with non-zero differences at cent precision, and one order without payment information. Total absolute discrepancies amount to R$3,233.00. Their underlying causes remain unconfirmed, and existing payment-quality flags overlap with only two discrepant orders.

Orders above R$400 represent approximately 4.82% of delivered orders but contribute 28.57% of merchandise GMV. The largest individual commercial segment comprises orders above R$150 and up to R$400 with positive freight burden up to 10%, contributing 16.38% of merchandise GMV.

Freight exceeds merchandise value for 3,075 orders. Of these, 2,919 have merchandise values up to R$50. These orders warrant further investigation into shipping economics, but recorded freight charges alone do not establish profitability or customer dissatisfaction.

Payment-method groups differ in review scores and delivery performance. Boleto-dominant orders have the highest observed late-delivery rate at 7.32%, while debit-card-dominant orders have the lowest at 5.33%. These descriptive differences may reflect product, regional, customer, or other characteristics.

### Recommended Follow-Up

Investigate the largest payment discrepancies against the underlying item and payment records before making corrections. Review low-value orders with high freight burden to understand the products, locations, and shipping arrangements involved.

Use the validated commercial segments to prioritize further analysis. Any proposed shipping or payment intervention requires additional evidence; this delivered-order dataset does not measure abandoned purchases, causal effects, or profit.

### Analytical Limitations

The analysis covers delivered orders only. Merchandise GMV excludes freight and is not net sales or profit. Dominant payment classification assigns each order to one method even when multiple methods were used.

Installment analysis uses payment-record grain, while commercial and customer-experience comparisons use order grain. Review and delivery measures use separate eligible populations, with missing values retained rather than replaced with zero.

Value and freight bands are descriptive analytical choices. Observed associations do not establish causation, and reconciliation flags identify differences rather than confirmed errors.

In [71]:
# Load the SQL dominant-payment-method summary before SQL–Python reconciliation.
sql_dominant_payment_summary = pd.read_csv("../reports/tables/sql_dominant_payment_summary.csv")

print("Shape:", sql_dominant_payment_summary.shape)
print("Columns:", sql_dominant_payment_summary.columns.tolist())
display(sql_dominant_payment_summary)

Shape: (4, 5)
Columns: ['dominant_payment_type', 'order_count', 'total_payment_value', 'average_payment_value_per_order', 'payment_value_share_pct']


,dominant_payment_type,order_count,total_payment_value,average_payment_value_per_order,payment_value_share_pct
0,credit_card,72785,12095049.25,166.175025,78.424893
1,boleto,19191,2769932.58,144.334978,17.960378
2,voucher,3017,349108.82,115.713894,2.263638
3,debit_card,1484,208371.12,140.411805,1.351088


In [72]:
# Combine existing Python dominant-payment summaries for SQL reconciliation.
python_dominant_payment_summary = dominant_payment_method_summary.merge(dominant_payment_value_summary, on='dominant_payment_type', validate='one_to_one')


In [73]:
# Calculate each dominant method's share of total order payment value.
python_dominant_payment_summary['payment_value_share_pct'] = (python_dominant_payment_summary['total_order_payment_value'] / python_dominant_payment_summary['total_order_payment_value'].sum()) * 100
python_dominant_payment_summary

,dominant_payment_type,order_count,order_share_pct,total_order_payment_value,mean_order_payment_value,median_order_payment_value,payment_value_share_pct
0,credit_card,72785,75.442852,12095049.25,166.175026,109.410,78.424894
1,boleto,19191,19.891788,2769932.58,144.334979,93.780,17.960379
2,voucher,3017,3.127170,349108.82,115.713895,81.950,2.263639
3,debit_card,1484,1.538190,208371.12,140.411806,89.625,1.351089


In [74]:
# Match SQL and Python dominant-payment summaries for reconciliation.
dominant_payment_comparison = sql_dominant_payment_summary.merge(python_dominant_payment_summary, on='dominant_payment_type', how='outer', validate='one_to_one', suffixes=('_sql', '_python'), indicator=True)

print("dominant_payment_comparison shape: ", dominant_payment_comparison.shape)
print("dominant_payment_comparison duplicated dominant_payment_type: ", dominant_payment_comparison['dominant_payment_type'].duplicated().sum())
print("dominant_payment_comparison merge indicator: ", dominant_payment_comparison['_merge'].value_counts())

dominant_payment_comparison shape:  (4, 12)
dominant_payment_comparison duplicated dominant_payment_type:  0
dominant_payment_comparison merge indicator:  _merge
both          4
left_only     0
right_only    0
Name: count, dtype: int64


In [75]:
# Validate SQL–Python dominant-payment counts, totals, averages, and value shares.
dominant_payment_validation = [
    {
        'SQL column' : 'order_count_sql',
        'PYTHON column' : 'order_count_python',
        'Comparison' : ('Exactly Equal'
                        if dominant_payment_comparison['order_count_sql'].eq(dominant_payment_comparison['order_count_python']).all()
                        else 'Not Equal')
    },
    {
        'SQL column' : 'total_payment_value',
        'PYTHON column' : 'total_order_payment_value',
        'Comparison' : ('Absolute difference below 0.01'
                        if (dominant_payment_comparison['total_payment_value'] - dominant_payment_comparison['total_order_payment_value']).abs().lt(0.01).all()
                        else 'Outside tolerance')
    },
    {
        'SQL column' : 'average_payment_value_per_order',
        'PYTHON column' : 'mean_order_payment_value',
        'Comparison' : ('Absolute difference below 0.01'
                        if (dominant_payment_comparison['average_payment_value_per_order'] - dominant_payment_comparison['mean_order_payment_value']).abs().lt(0.01).all()
                        else 'Outside tolerance')
    },
    {
            'SQL column' : 'payment_value_share_pct_sql',
            'PYTHON column' : 'payment_value_share_pct_python',
            'Comparison' : ('Absolute difference below 0.00001'
                            if (dominant_payment_comparison['payment_value_share_pct_sql'] - dominant_payment_comparison['payment_value_share_pct_python']).abs().lt(0.00001).all()
                            else 'Outside tolerance')
    }
]

dominant_payment_validation_summary = pd.DataFrame(dominant_payment_validation)
dominant_payment_validation_summary

,SQL column,PYTHON column,Comparison
0,order_count_sql,order_count_python,Exactly Equal
1,total_payment_value,total_order_payment_value,Absolute difference below 0.01
2,average_payment_value_per_order,mean_order_payment_value,Absolute difference below 0.01
3,payment_value_share_pct_sql,payment_value_share_pct_python,Absolute difference below 0.00001


### SQL–Python Dominant-Payment-Method Reconciliation Summary

- All four dominant payment methods matched, with no duplicate or unmatched methods.
- Order counts matched exactly, covering 96,477 delivered orders with payment records.
- Total and average order payment values matched within R$0.01.
- Payment-value shares differed by less than 0.00001 percentage points.
- Total payments reconciled to R$15,422,461.77.
- Each order was classified by its highest-value payment method.
  Grouped payment totals include all payment methods used for those orders.
- The single delivered order without payment records was excluded
  from this analysis and documented separately.

In [76]:
# Load the SQL installment summary and inspect its structure before SQL–Python reconciliation.
sql_installment_summary = pd.read_csv("../reports/tables/sql_installment_summary.csv")

print("Shape:", sql_installment_summary.shape)
print("Columns:", sql_installment_summary.columns.tolist())
display(sql_installment_summary)

Shape: (5, 6)
Columns: ['installment_group', 'payment_record_count', 'total_payment_value', 'average_payment_value', 'payment_record_share_pct', 'median_payment_value']


,installment_group,payment_record_count,total_payment_value,average_payment_value,payment_record_share_pct,median_payment_value
0,Single installment,24759,2366960.73,95.600013,33.196128,67.74
1,2–3 installments,22239,2971869.74,133.633245,29.817387,109.89
2,4–6 installments,15790,2849773.90,180.479664,21.170760,127.01
3,7–12 installments,11617,3838346.31,330.407705,15.575727,204.41
4,13–24 installments,179,73955.57,413.159608,0.239998,252.09


In [77]:
# Inspect the existing Python installment summary before SQL–Python reconciliation.
installment_group_summary

,installment_group,payment_record_count,mean_payment_value,median_payment_value,payment_record_share_pct
0,Single installment,24759,95.600013,67.74,33.196128
1,2-3 installments,22239,133.633245,109.89,29.817387
2,4-6 installments,15790,180.479664,127.01,21.170760
3,7-12 installments,11617,330.407705,204.41,15.575727
4,13-24 installments,179,413.159609,252.09,0.239998


In [78]:
# Summarize valid credit-card payment counts, totals, averages, and medians by installment group.
installment_group_summary = (valid_credit_card_payments.groupby('installment_group', observed=False).agg(payment_record_count=('payment_value', 'size'),
                                                                                                         total_payment_value=('payment_value', 'sum'),
                                                                                                         mean_payment_value=('payment_value', 'mean'),
                                                                                                         median_payment_value=('payment_value', 'median')
                                                                                                         ).reset_index())

# Calculate each group's share of valid credit-card payment records.
installment_group_summary['payment_record_share_pct'] = (installment_group_summary['payment_record_count']/ len(valid_credit_card_payments)* 100)

display(installment_group_summary)

,installment_group,payment_record_count,total_payment_value,mean_payment_value,median_payment_value,payment_record_share_pct
0,Single installment,24759,2366960.73,95.600013,67.74,33.196128
1,2-3 installments,22239,2971869.74,133.633245,109.89,29.817387
2,4-6 installments,15790,2849773.90,180.479664,127.01,21.170760
3,7-12 installments,11617,3838346.31,330.407705,204.41,15.575727
4,13-24 installments,179,73955.57,413.159609,252.09,0.239998


In [79]:
# Standardize SQL installment-group labels to match the existing Python summary.
sql_installment_summary['installment_group'] = (sql_installment_summary['installment_group'].str.replace('_', '-', regex=False))

display(sql_installment_summary['installment_group'])

0    Single installment
1      2–3 installments
2      4–6 installments
3     7–12 installments
4    13–24 installments
Name: installment_group, dtype: str

In [80]:
# Normalize dash characters in SQL installment-group labels.
sql_installment_summary['installment_group'] = (
    sql_installment_summary['installment_group']
    .str.replace('[\u2010-\u2015\u2212]', '-', regex=True)
)

display(sql_installment_summary['installment_group'])

0    Single installment
1      2-3 installments
2      4-6 installments
3     7-12 installments
4    13-24 installments
Name: installment_group, dtype: str

In [81]:
# Match SQL and Python installment summaries for reconciliation.
installment_comparison = sql_installment_summary.merge(installment_group_summary, on='installment_group', how='outer', validate='one_to_one', suffixes=('_sql', '_python'),indicator=True)

# Validate installment-group coverage and uniqueness.
print("Shape:", installment_comparison.shape)
print("Duplicate installment groups:", installment_comparison['installment_group'].duplicated().sum())
print("Merge indicator counts:\n", installment_comparison['_merge'].value_counts())

display(installment_comparison)

Shape: (5, 12)
Duplicate installment groups: 0
Merge indicator counts:
 _merge
both          5
left_only     0
right_only    0
Name: count, dtype: int64


,installment_group,payment_record_count_sql,total_payment_value_sql,average_payment_value,payment_record_share_pct_sql,median_payment_value_sql,payment_record_count_python,total_payment_value_python,mean_payment_value,median_payment_value_python,payment_record_share_pct_python,_merge
0,13-24 installments,179,73955.57,413.159608,0.239998,252.09,179,73955.57,413.159609,252.09,0.239998,both
1,2-3 installments,22239,2971869.74,133.633245,29.817387,109.89,22239,2971869.74,133.633245,109.89,29.817387,both
2,4-6 installments,15790,2849773.90,180.479664,21.170760,127.01,15790,2849773.90,180.479664,127.01,21.170760,both
3,7-12 installments,11617,3838346.31,330.407705,15.575727,204.41,11617,3838346.31,330.407705,204.41,15.575727,both
4,Single installment,24759,2366960.73,95.600013,33.196128,67.74,24759,2366960.73,95.600013,67.74,33.196128,both


In [82]:
# Validate SQL–Python installment counts, payment totals, averages, medians, and record shares.
installment_validation = [
    {
        'SQL column': 'payment_record_count_sql',
        'Python column': 'payment_record_count_python',
        'Comparison': ('Exactly equal' 
                       if installment_comparison['payment_record_count_sql'].eq(installment_comparison['payment_record_count_python']).all()
                       else 'Not equal')
    },
    {
        'SQL column': 'total_payment_value_sql',
        'Python column': 'total_payment_value_python',
        'Comparison': ('Absolute difference below 0.01'
                       if (installment_comparison['total_payment_value_sql'] - installment_comparison['total_payment_value_python']).abs().lt(0.01).all()
                       else 'Outside tolerance'
                       )
    },
    {
        'SQL column': 'average_payment_value',
        'Python column': 'mean_payment_value',
        'Comparison': ('Absolute difference below 0.01'
                       if (installment_comparison['average_payment_value'] - installment_comparison['mean_payment_value'] ).abs().lt(0.01).all()
                       else 'Outside tolerance')
    },
    {
        'SQL column': 'median_payment_value_sql',
        'Python column': 'median_payment_value_python',
        'Comparison': ('Absolute difference below 0.01'
                       if (installment_comparison['median_payment_value_sql'] - installment_comparison['median_payment_value_python']).abs().lt(0.01).all()
                       else 'Outside tolerance'
        )
    },
    {
        'SQL column': 'payment_record_share_pct_sql',
        'Python column': 'payment_record_share_pct_python',
        'Comparison': ('Absolute difference below 0.00001 percentage points'
                       if (installment_comparison['payment_record_share_pct_sql'] - installment_comparison['payment_record_share_pct_python']).abs().lt(0.00001).all()
                       else 'Outside tolerance')
    }
]

# Display the installment validation summary.
installment_validation_summary = pd.DataFrame(installment_validation)
display(installment_validation_summary)

,SQL column,Python column,Comparison
0,payment_record_count_sql,payment_record_count_python,Exactly equal
1,total_payment_value_sql,total_payment_value_python,Absolute difference below 0.01
2,average_payment_value,mean_payment_value,Absolute difference below 0.01
3,median_payment_value_sql,median_payment_value_python,Absolute difference below 0.01
4,payment_record_share_pct_sql,payment_record_share_pct_python,Absolute difference below 0.00001 percentage p...


### Credit-card installment analysis — SQL–Python reconciliation

SQL and Python results reconciled successfully across all **five installment groups**.

- All five groups matched, with no duplicate or unmatched groups.
- Payment-record counts matched exactly, covering **74,584 valid credit-card payment records** from delivered orders.
- Total payment values, average payment values, and median payment values differed by less than **R$0.01**.
- Payment-record shares differed by less than **0.00001 percentage points**.
- **Two records with zero installments** were excluded from installment analysis.

**Business findings**

- Single-installment payments represented the largest group: **24,759 records (33.20%)**.
- Payments using **2–3 installments** accounted for **22,239 records (29.82%)**.
- Median payment value increased across installment groups, from **R$67.74** for single installments to **R$252.09** for 13–24 installments.
- The 13–24 installment group contained only **179 records (0.24%)**, so its results describe a small segment.

These measures describe **payment records, not unique orders**. The relationship between installment count and payment value is descriptive and does not establish causation.

In [83]:
# Load the SQL payment-reconciliation summary and inspect its structure.
sql_payment_reconciliation_summary = pd.read_csv("../reports/tables/sql_payment_reconciliation_summary.csv")

print("Shape:", sql_payment_reconciliation_summary.shape)
print("Columns:", sql_payment_reconciliation_summary.columns.tolist())
display(sql_payment_reconciliation_summary)

Shape: (4, 5)
Columns: ['payment_reconciliation_status', 'order_count', 'order_share_pct', 'total_payment_difference', 'total_absolute_payment_difference']


,payment_reconciliation_status,order_count,order_share_pct,total_payment_difference,total_absolute_payment_difference
0,matched,95908,99.409192,0.00,0.00
1,missing_payment,1,0.001037,NaN,NaN
2,payment_above_expected,362,0.375215,3032.24,3032.24
3,payment_below_expected,207,0.214557,-200.76,200.76


In [84]:
# Summarize Python payment-reconciliation results by status for SQL comparison.
python_payment_reconciliation_summary = (order_payment_reconciliation.groupby('payment_reconciliation_status').agg(order_count=('order_id', 'size'),
                                                                                                                   total_payment_difference=('payment_difference',lambda x: x.sum(min_count=1)),
                                                                                                                   total_absolute_payment_difference=('absolute_payment_difference',lambda x: x.sum(min_count=1))).reset_index())

# Calculate each status's share of all delivered orders.
python_payment_reconciliation_summary['order_share_pct'] = (python_payment_reconciliation_summary['order_count'] / len(order_payment_reconciliation)* 100)

# Display columns in the same order as the SQL export.
python_payment_reconciliation_summary = (python_payment_reconciliation_summary[
    [
        'payment_reconciliation_status',
        'order_count',
        'order_share_pct',
        'total_payment_difference',
        'total_absolute_payment_difference'
    ]
])

print("Shape:", python_payment_reconciliation_summary.shape)
display(python_payment_reconciliation_summary)

Shape: (4, 5)


,payment_reconciliation_status,order_count,order_share_pct,total_payment_difference,total_absolute_payment_difference
0,matched,95908,99.409192,-3.742784e-11,4.041922e-10
1,missing_payment,1,0.001037,NaN,NaN
2,payment_above_expected,362,0.375215,3.032240e+03,3.032240e+03
3,payment_below_expected,207,0.214557,-2.007600e+02,2.007600e+02


In [85]:
# Match SQL and Python payment-reconciliation summaries.
payment_reconciliation_comparison = sql_payment_reconciliation_summary.merge(python_payment_reconciliation_summary,
                                                                             on='payment_reconciliation_status',
                                                                             how='outer',
                                                                             validate='one_to_one',
                                                                             suffixes=('_sql', '_python'),
                                                                             indicator=True)

# Validate status coverage and uniqueness.
print("Shape:", payment_reconciliation_comparison.shape)
print("Duplicate statuses:", payment_reconciliation_comparison['payment_reconciliation_status'].duplicated().sum())
print("Merge indicator counts:\n",payment_reconciliation_comparison['_merge'].value_counts())

display(payment_reconciliation_comparison)

Shape: (4, 10)
Duplicate statuses: 0
Merge indicator counts:
 _merge
both          4
left_only     0
right_only    0
Name: count, dtype: int64


,payment_reconciliation_status,order_count_sql,order_share_pct_sql,total_payment_difference_sql,total_absolute_payment_difference_sql,order_count_python,order_share_pct_python,total_payment_difference_python,total_absolute_payment_difference_python,_merge
0,matched,95908,99.409192,0.00,0.00,95908,99.409192,-3.742784e-11,4.041922e-10,both
1,missing_payment,1,0.001037,NaN,NaN,1,0.001037,NaN,NaN,both
2,payment_above_expected,362,0.375215,3032.24,3032.24,362,0.375215,3.032240e+03,3.032240e+03,both
3,payment_below_expected,207,0.214557,-200.76,200.76,207,0.214557,-2.007600e+02,2.007600e+02,both


In [86]:
# Validate SQL–Python reconciliation counts, shares, differences, and missing-value consistency.
payment_reconciliation_validation = [
    {
        'SQL column': 'order_count_sql',
        'Python column': 'order_count_python',
        'Comparison': ('Exactly equal'
                       if payment_reconciliation_comparison['order_count_sql'].eq(payment_reconciliation_comparison['order_count_python']).fillna(False).all()
                       else 'Not equal'
                       )
    },
    {
        'SQL column': 'order_share_pct_sql',
        'Python column': 'order_share_pct_python',
        'Comparison': ('Absolute difference below 0.00001 percentage points'
                       if (payment_reconciliation_comparison['order_share_pct_sql'] - payment_reconciliation_comparison['order_share_pct_python']).abs().lt(0.00001).fillna(False).all()
                       else 'Outside tolerance'
                       )
    }
]

# Compare monetary totals, accepting missing values only when both sides are missing.
for column in ['total_payment_difference', 'total_absolute_payment_difference']:
    sql_values = payment_reconciliation_comparison[f'{column}_sql']
    python_values = payment_reconciliation_comparison[f'{column}_python']

    both_missing = sql_values.isna() & python_values.isna()
    both_present = sql_values.notna() & python_values.notna()
    within_tolerance = ((sql_values - python_values).abs().lt(0.01).fillna(False))

    matches = both_missing | (both_present & within_tolerance)

    payment_reconciliation_validation.append({'SQL column': f'{column}_sql',
                                              'Python column': f'{column}_python',
                                              'Comparison': ('Difference below R$0.01; missing values match'
                                                             if matches.all()
                                                             else 'Outside tolerance or missing-value mismatch'
                                                             )
                                            })

# Display the validation summary.
payment_reconciliation_validation_summary = pd.DataFrame(payment_reconciliation_validation)

display(payment_reconciliation_validation_summary)

,SQL column,Python column,Comparison
0,order_count_sql,order_count_python,Exactly equal
1,order_share_pct_sql,order_share_pct_python,Absolute difference below 0.00001 percentage p...
2,total_payment_difference_sql,total_payment_difference_python,Difference below R$0.01; missing values match
3,total_absolute_payment_difference_sql,total_absolute_payment_difference_python,Difference below R$0.01; missing values match


### Payment-value reconciliation — SQL–Python validation

SQL and Python results reconciled across all four payment-reconciliation statuses.

- All four statuses matched, with no duplicates or unmatched statuses.
- Order counts matched exactly across **96,478 delivered orders**.
- Order shares differed by less than **0.00001 percentage points**.
- Signed and absolute payment-difference totals differed by less than **R$0.01**.
- Missing payment differences remained missing in both SQL and Python.

**Findings**

- **95,908 orders (99.41%)** matched merchandise plus freight after rounding payment differences to two decimal places.
- **362 orders** had recorded payments above expected value, with combined positive differences of **R$3,032.24**.
- **207 orders** had recorded payments below expected value, with combined shortfalls of **R$200.76**.
- The **569 orders with discrepancies** had total absolute differences of **R$3,233.00**.
- **One delivered order**, with expected value **R$143.46**, had no payment data. It was retained and classified separately as `missing_payment`.

These classifications describe differences between recorded amounts; they do not establish their underlying cause. Tiny floating-point residuals in Python were handled through rounding for classification and tolerances for reconciliation.

In [87]:
# Load the SQL order-value-band summary and inspect its structure before SQL–Python reconciliation.
sql_order_value_band_summary = pd.read_csv("../reports/tables/sql_order_value_band_summary.csv")

print("Shape:", sql_order_value_band_summary.shape)
print("Columns:", sql_order_value_band_summary.columns.tolist())
display(sql_order_value_band_summary)

Shape: (6, 6)
Columns: ['order_value_band', 'order_count', 'total_merchandise_value', 'average_merchandise_value', 'order_share_pct', 'merchandise_value_share_pct']


,order_value_band,order_count,total_merchandise_value,average_merchandise_value,order_share_pct,merchandise_value_share_pct
0,0-50,29045,929291.71,31.994894,30.105309,7.028641
1,50-100,27606,2092612.98,75.802831,28.613777,15.827351
2,100-150,16501,2086383.49,126.439821,17.103381,15.780235
3,150-400,18675,4335987.80,232.181408,19.356745,32.794981
4,400-1000,3746,2312659.72,617.367784,3.882750,17.491661
5,1000+,905,1464562.41,1618.301005,0.938038,11.077129


In [88]:
# Inspect the existing Python order-value-band summary before SQL–Python reconciliation.
display(order_value_band_summary)

,order_value_band,order_count,merchandise_gmv,order_share_pct,gmv_share_pct
0,0-50,29045,929291.71,30.105309,7.028642
1,50-100,27606,2092612.98,28.613777,15.827352
2,100-150,16501,2086383.49,17.103381,15.780235
3,150-400,18675,4335987.80,19.356745,32.794981
4,400-1000,3746,2312659.72,3.882750,17.491662
5,1000+,905,1464562.41,0.938038,11.077129


In [89]:
# Prepare the Python order-value-band summary and calculate average merchandise value per order.
python_order_value_band_summary = order_value_band_summary.copy()

python_order_value_band_summary['average_merchandise_value'] = (python_order_value_band_summary['merchandise_gmv'] / python_order_value_band_summary['order_count'])

display(python_order_value_band_summary)

,order_value_band,order_count,merchandise_gmv,order_share_pct,gmv_share_pct,average_merchandise_value
0,0-50,29045,929291.71,30.105309,7.028642,31.994894
1,50-100,27606,2092612.98,28.613777,15.827352,75.802832
2,100-150,16501,2086383.49,17.103381,15.780235,126.439821
3,150-400,18675,4335987.80,19.356745,32.794981,232.181408
4,400-1000,3746,2312659.72,3.882750,17.491662,617.367784
5,1000+,905,1464562.41,0.938038,11.077129,1618.301006


In [90]:
# Match SQL and Python order-value-band summaries for reconciliation.
order_value_band_comparison = sql_order_value_band_summary.merge(python_order_value_band_summary, 
                                                                 on='order_value_band',
                                                                 how='outer',
                                                                 validate='one_to_one',
                                                                 suffixes=('_sql', '_python'),
                                                                 indicator=True
                                                                 )

print("Shape:", order_value_band_comparison.shape)
print("Duplicate bands:", order_value_band_comparison['order_value_band'].duplicated().sum())
print("Merge indicator counts:", order_value_band_comparison['_merge'].value_counts(), sep="\n")

display(order_value_band_comparison)

Shape: (6, 12)
Duplicate bands: 0
Merge indicator counts:
_merge
both          6
left_only     0
right_only    0
Name: count, dtype: int64


,order_value_band,order_count_sql,total_merchandise_value,average_merchandise_value_sql,order_share_pct_sql,merchandise_value_share_pct,order_count_python,merchandise_gmv,order_share_pct_python,gmv_share_pct,average_merchandise_value_python,_merge
0,0-50,29045,929291.71,31.994894,30.105309,7.028641,29045,929291.71,30.105309,7.028642,31.994894,both
1,100-150,16501,2086383.49,126.439821,17.103381,15.780235,16501,2086383.49,17.103381,15.780235,126.439821,both
2,1000+,905,1464562.41,1618.301005,0.938038,11.077129,905,1464562.41,0.938038,11.077129,1618.301006,both
3,150-400,18675,4335987.80,232.181408,19.356745,32.794981,18675,4335987.80,19.356745,32.794981,232.181408,both
4,400-1000,3746,2312659.72,617.367784,3.882750,17.491661,3746,2312659.72,3.882750,17.491662,617.367784,both
5,50-100,27606,2092612.98,75.802831,28.613777,15.827351,27606,2092612.98,28.613777,15.827352,75.802832,both


In [91]:
# Validate SQL–Python order-value-band counts, values, averages, and percentage shares.
comparison_checks = [
    ('order_count_sql', 'order_count_python', None),
    ('total_merchandise_value', 'merchandise_gmv', 0.01),
    ('average_merchandise_value_sql', 'average_merchandise_value_python', 0.01),
    ('order_share_pct_sql', 'order_share_pct_python', 0.00001),
    ('merchandise_value_share_pct', 'gmv_share_pct', 0.00001)
]

order_value_band_validation = []

for sql_column, python_column, tolerance in comparison_checks:
    sql_values = order_value_band_comparison[sql_column]
    python_values = order_value_band_comparison[python_column]

    if tolerance is None:
        passed = sql_values.eq(python_values).fillna(False).all()
        result = 'Exactly equal' if passed else 'Not equal'
    else:
        passed = (
            (sql_values - python_values)
            .abs()
            .lt(tolerance)
            .fillna(False)
            .all()
        )
        result = (
            f'Absolute difference below {tolerance:.5f} percentage points'
            if tolerance == 0.00001
            else 'Absolute difference below R$0.01'
        ) if passed else 'Outside tolerance'

    order_value_band_validation.append({
        'SQL column': sql_column,
        'Python column': python_column,
        'Comparison': result
    })

order_value_band_validation_summary = pd.DataFrame(order_value_band_validation)

display(order_value_band_validation_summary)

,SQL column,Python column,Comparison
0,order_count_sql,order_count_python,Exactly equal
1,total_merchandise_value,merchandise_gmv,Absolute difference below R$0.01
2,average_merchandise_value_sql,average_merchandise_value_python,Absolute difference below R$0.01
3,order_share_pct_sql,order_share_pct_python,Absolute difference below 0.00001 percentage p...
4,merchandise_value_share_pct,gmv_share_pct,Absolute difference below 0.00001 percentage p...


### SQL–Python Order-Value-Band Reconciliation

- All six merchandise-value bands matched, with no duplicate or unmatched bands.
- Order counts matched exactly across all bands.
- Merchandise totals and average merchandise values differed by less than R$0.01.
- Order shares and merchandise-value shares differed by less than 0.00001 percentage points.
- The summary covers 96,478 delivered orders and R$13,221,498.11 in merchandise value.

Orders worth R$100 or less represent 58.72% of delivered orders and 22.86% of merchandise value. The R$150–400 band contributes the largest merchandise-value share at 32.79%.

In [92]:
# Load the SQL freight-burden summary and inspect its structure before SQL–Python reconciliation.
sql_freight_burden_summary = pd.read_csv("../reports/tables/sql_freight_burden_summary.csv")

print("Shape:", sql_freight_burden_summary.shape)
print("Columns:", sql_freight_burden_summary.columns.tolist())
display(sql_freight_burden_summary)

Shape: (6, 6)
Columns: ['freight_burden_band', 'order_count', 'total_merchandise_value', 'total_freight_value', 'average_freight_burden_pct', 'order_share_pct']


,freight_burden_band,order_count,total_merchandise_value,total_freight_value,average_freight_burden_pct,order_share_pct
0,Zero freight,336,37531.57,0.00,0.000000,0.348266
1,0-10%,14925,5572645.23,301859.40,6.688300,15.469848
2,10-25%,38168,5032721.48,811505.70,17.022780,39.561351
3,25-50%,27778,2034799.10,686316.65,34.719303,28.792056
4,50-100%,12196,478179.36,308591.50,66.866178,12.641224
5,Above 100%,3075,65621.37,90002.39,144.914270,3.187255


In [93]:
# Inspect the existing Python freight-burden summary before SQL–Python reconciliation.
display(freight_burden_summary)

,freight_burden_band,order_count,merchandise_gmv,total_freight_value,order_share_pct,gmv_share_pct,freight_share_pct
0,Zero freight,336,37531.57,0.00,0.348266,0.283868,0.000000
1,0-10%,14925,5572645.23,301859.40,15.469848,42.148365,13.731645
2,10-25%,38168,5032721.48,811505.70,39.561351,38.064684,36.915557
3,25-50%,27778,2034799.10,686316.65,28.792056,15.390080,31.220682
4,50-100%,12196,478179.36,308591.50,12.641224,3.616681,14.037889
5,Above 100%,3075,65621.37,90002.39,3.187255,0.496323,4.094227


In [94]:
# Add average order-level freight burden to the Python summary for SQL reconciliation.
python_freight_burden_averages = order_commercial_summary.groupby('freight_burden_band').agg(average_freight_burden_pct = ('freight_burden_pct', 'mean')).reset_index()

python_freight_burden_summary = freight_burden_summary.merge(python_freight_burden_averages, on='freight_burden_band', how='left', validate='one_to_one')

display(python_freight_burden_summary)

,freight_burden_band,order_count,merchandise_gmv,total_freight_value,order_share_pct,gmv_share_pct,freight_share_pct,average_freight_burden_pct
0,Zero freight,336,37531.57,0.00,0.348266,0.283868,0.000000,0.000000
1,0-10%,14925,5572645.23,301859.40,15.469848,42.148365,13.731645,6.688301
2,10-25%,38168,5032721.48,811505.70,39.561351,38.064684,36.915557,17.022781
3,25-50%,27778,2034799.10,686316.65,28.792056,15.390080,31.220682,34.719304
4,50-100%,12196,478179.36,308591.50,12.641224,3.616681,14.037889,66.866179
5,Above 100%,3075,65621.37,90002.39,3.187255,0.496323,4.094227,144.914271


In [95]:
# Match SQL and Python freight-burden summaries for reconciliation.
freight_burden_comparison = sql_freight_burden_summary.merge(python_freight_burden_summary,
                                                             on='freight_burden_band',
                                                             how='outer',
                                                             validate='one_to_one',
                                                             suffixes=('_sql', '_python'),
                                                             indicator=True,
                                                             )

display(freight_burden_comparison)

,freight_burden_band,order_count_sql,total_merchandise_value,total_freight_value_sql,average_freight_burden_pct_sql,order_share_pct_sql,order_count_python,merchandise_gmv,total_freight_value_python,order_share_pct_python,gmv_share_pct,freight_share_pct,average_freight_burden_pct_python,_merge
0,0-10%,14925,5572645.23,301859.40,6.688300,15.469848,14925,5572645.23,301859.40,15.469848,42.148365,13.731645,6.688301,both
1,10-25%,38168,5032721.48,811505.70,17.022780,39.561351,38168,5032721.48,811505.70,39.561351,38.064684,36.915557,17.022781,both
2,25-50%,27778,2034799.10,686316.65,34.719303,28.792056,27778,2034799.10,686316.65,28.792056,15.390080,31.220682,34.719304,both
3,50-100%,12196,478179.36,308591.50,66.866178,12.641224,12196,478179.36,308591.50,12.641224,3.616681,14.037889,66.866179,both
4,Above 100%,3075,65621.37,90002.39,144.914270,3.187255,3075,65621.37,90002.39,3.187255,0.496323,4.094227,144.914271,both
5,Zero freight,336,37531.57,0.00,0.000000,0.348266,336,37531.57,0.00,0.348266,0.283868,0.000000,0.000000,both


In [96]:
# Validate SQL–Python freight-burden counts, monetary totals, average burden, and order shares.
comparison_checks = [
    ('order_count_sql', 'order_count_python', None),
    ('total_merchandise_value', 'merchandise_gmv', 0.01),
    ('total_freight_value_sql', 'total_freight_value_python', 0.01),
    ('average_freight_burden_pct_sql', 'average_freight_burden_pct_python', 0.00001),
    ('order_share_pct_sql', 'order_share_pct_python', 0.00001)
]

freight_burden_validation = []

for sql_column, python_column, tolerance in comparison_checks:
    sql_values = freight_burden_comparison[sql_column]
    python_values = freight_burden_comparison[python_column]

    if tolerance is None:
        passed = sql_values.eq(python_values).fillna(False).all()
        result = 'Exactly equal' if passed else 'Not equal'
    else:
        passed = ((sql_values - python_values).abs().lt(tolerance).fillna(False).all())
        result = (f'Absolute difference below {tolerance:.5f} percentage points'
                  if tolerance == 0.00001
                  else 'Absolute difference below R$0.01'
                  ) if passed else 'Outside tolerance'

    freight_burden_validation.append({
        'SQL column': sql_column,
        'Python column': python_column,
        'Comparison': result
    })

freight_burden_validation_summary = pd.DataFrame(freight_burden_validation)

display(freight_burden_validation_summary)

,SQL column,Python column,Comparison
0,order_count_sql,order_count_python,Exactly equal
1,total_merchandise_value,merchandise_gmv,Absolute difference below R$0.01
2,total_freight_value_sql,total_freight_value_python,Absolute difference below R$0.01
3,average_freight_burden_pct_sql,average_freight_burden_pct_python,Absolute difference below 0.00001 percentage p...
4,order_share_pct_sql,order_share_pct_python,Absolute difference below 0.00001 percentage p...


### SQL–Python Freight-Burden Reconciliation

- All six freight-burden bands matched, with no unmatched bands.
- Order counts matched exactly.
- Merchandise and freight totals differed by less than R$0.01.
- Average freight burden and order shares differed by less than 0.00001 percentage points.
- The summary covers 96,478 delivered orders, R$13,221,498.11 in merchandise value, and R$2,198,275.64 in freight.

The largest band was freight above 10% and up to 25% of merchandise value, covering 39.56% of orders. Freight exceeded merchandise value for 3,075 orders (3.19%), while 336 orders (0.35%) had zero freight.

Average freight burden is the mean of individual order percentages. These bands describe recorded freight charges relative to merchandise value; they do not establish the reasons for high charges.

In [97]:
# Load the SQL commercial-segment summary and inspect its structure before SQL–Python reconciliation.
sql_commercial_segment_summary = pd.read_csv('../reports/tables/sql_commercial_segment_summary.csv')

print("Shape:", sql_commercial_segment_summary.shape)
print("Columns:", sql_commercial_segment_summary.columns.tolist())
display(sql_commercial_segment_summary)

Shape: (33, 5)
Columns: ['order_value_band', 'freight_burden_band', 'order_count', 'merchandise_gmv', 'total_freight_value']


,order_value_band,freight_burden_band,order_count,merchandise_gmv,total_freight_value
0,150-400,0-10%,8679,2166289.99,148380.44
1,400-1000,0-10%,2806,1758930.22,82535.10
2,150-400,10-25%,7905,1713663.62,261577.84
3,100-150,10-25%,11122,1402281.87,217678.23
4,1000+,0-10%,807,1329625.91,44561.43
5,50-100,10-25%,15398,1220966.02,223744.02
6,50-100,25-50%,9723,678046.97,222204.32
7,0-50,25-50%,13194,492580.57,178985.13
8,400-1000,10-25%,777,458975.27,67538.52
9,150-400,25-50%,1806,395101.62,129482.55


In [98]:
# Match SQL and Python commercial-segment summaries using both band columns.
commercial_segment_comparison = sql_commercial_segment_summary.merge(commercial_segment_summary,
                                                                     on=['order_value_band', 'freight_burden_band'],
                                                                     how='outer',
                                                                     validate='one_to_one',
                                                                     suffixes=('_sql', '_python'),
                                                                     indicator=True
                                                                     )

print("Shape:", commercial_segment_comparison.shape)
print("Duplicate band combinations:", commercial_segment_comparison.duplicated(subset=['order_value_band', 'freight_burden_band']).sum())
print("Merge indicator counts:", commercial_segment_comparison['_merge'].value_counts(), sep="\n")

display(commercial_segment_comparison)

Shape: (33, 12)
Duplicate band combinations: 0
Merge indicator counts:
_merge
both          33
left_only      0
right_only     0
Name: count, dtype: int64


,order_value_band,freight_burden_band,order_count_sql,merchandise_gmv_sql,total_freight_value_sql,order_count_python,merchandise_gmv_python,total_freight_value_python,order_share_pct,gmv_share_pct,freight_share_pct,_merge
0,0-50,10-25%,2887,124953.54,25018.47,2887,124953.54,25018.47,2.992392,0.945079,1.138095,both
1,0-50,25-50%,13194,492580.57,178985.13,13194,492580.57,178985.13,13.675657,3.725603,8.142069,both
2,0-50,50-100%,10045,260177.01,172531.48,10045,260177.01,172531.48,10.411700,1.967833,7.848492,both
3,0-50,Above 100%,2919,51580.59,71244.26,2919,51580.59,71244.26,3.025560,0.390127,3.240916,both
4,100-150,0-10%,1933,254123.11,20608.79,1933,254123.11,20608.79,2.003566,1.922045,0.937498,both
5,100-150,10-25%,11122,1402281.87,217678.23,11122,1402281.87,217678.23,11.528017,10.606074,9.902226,both
6,100-150,25-50%,2891,362411.08,120623.89,2891,362411.08,120623.89,2.996538,2.741074,5.487205,both
7,100-150,50-100%,430,52863.83,32851.25,430,52863.83,32851.25,0.445697,0.399832,1.494410,both
8,100-150,Above 100%,33,4031.93,4585.92,33,4031.93,4585.92,0.034205,0.030495,0.208614,both
9,100-150,Zero freight,92,10671.67,0.00,92,10671.67,0.00,0.095359,0.080715,0.000000,both


In [99]:
# Validate SQL–Python commercial-segment order counts and monetary totals.
comparison_checks = [
    ('order_count_sql', 'order_count_python', None),
    ('merchandise_gmv_sql', 'merchandise_gmv_python', 0.01),
    ('total_freight_value_sql', 'total_freight_value_python', 0.01)
]

commercial_segment_validation = []

for sql_column, python_column, tolerance in comparison_checks:
    sql_values = commercial_segment_comparison[sql_column]
    python_values = commercial_segment_comparison[python_column]

    if tolerance is None:
        passed = sql_values.eq(python_values).fillna(False).all()
        result = 'Exactly equal' if passed else 'Not equal'
    else:
        passed = ((sql_values - python_values).abs().lt(tolerance).fillna(False).all())
        result = ('Absolute difference below R$0.01'
                  if passed else 'Outside tolerance'
                  )

    commercial_segment_validation.append({
        'SQL column': sql_column,
        'Python column': python_column,
        'Comparison': result
    })

commercial_segment_validation_summary = pd.DataFrame(commercial_segment_validation)

display(commercial_segment_validation_summary)

,SQL column,Python column,Comparison
0,order_count_sql,order_count_python,Exactly equal
1,merchandise_gmv_sql,merchandise_gmv_python,Absolute difference below R$0.01
2,total_freight_value_sql,total_freight_value_python,Absolute difference below R$0.01


### SQL–Python Commercial-Segment Reconciliation

- All 33 observed combinations of order-value and freight-burden bands matched.
- No duplicate or unmatched band combinations were found.
- Order counts matched exactly.
- Merchandise and freight totals differed by less than R$0.01 for every segment.
- The summary covers 96,478 delivered orders, R$13,221,498.11 in merchandise value, and R$2,198,275.64 in freight.

The largest segment by merchandise value combined orders above R$150 and up to R$400 with freight above 0% and up to 10% of merchandise value. It contained 8,679 orders and contributed R$2,166,289.99 in merchandise value.

These segments describe observed combinations of order value and freight burden; they do not establish profitability.

### SQL–Python Reconciliation: Payment-Method Experience Summary

Compare the SQL results with the Python summary for the four dominant payment methods across 96,477 delivered orders with payment records.

Then validate:
- Each payment method appears exactly once in both summaries.
- Order counts, eligible-order counts, and late-order counts match exactly.
- Merchandise values, freight burden, review scores, delivery durations, and late-delivery rates agree within the chosen tolerances.
- Each late-delivery rate uses only orders with a known delivery status for that payment method.

In [100]:
# Load the SQL payment-method experience summary and inspect its shape and columns before comparing it with the Python summary.
sql_payment_method_experience_summary = pd.read_csv('../reports/tables/sql_payment_method_experience_summary.csv')

print("Shape:", sql_payment_method_experience_summary.shape)
print("Columns:", sql_payment_method_experience_summary.columns.tolist())
display(sql_payment_method_experience_summary)

Shape: (4, 12)
Columns: ['dominant_payment_type', 'order_count', 'mean_merchandise_value', 'median_merchandise_value', 'median_freight_burden_pct', 'review_eligible_order_count', 'mean_order_review_score', 'delivery_duration_eligible_order_count', 'mean_delivery_duration_days', 'delivery_status_eligible_order_count', 'late_order_count', 'late_delivery_rate_pct']


,dominant_payment_type,order_count,mean_merchandise_value,median_merchandise_value,median_freight_burden_pct,review_eligible_order_count,mean_order_review_score,delivery_duration_eligible_order_count,mean_delivery_duration_days,delivery_status_eligible_order_count,late_order_count,late_delivery_rate_pct
0,boleto,19191,121.401054,74.90,25.282528,19062,4.156069,19191,13.491950,19191,1405,7.321140
1,credit_card,72785,143.235028,89.90,21.611111,72298,4.155658,72778,12.349937,72778,4869,6.690209
2,debit_card,1484,119.857452,70.45,24.905890,1478,4.241880,1483,10.788641,1483,79,5.327040
3,voucher,3017,95.566947,59.99,27.398058,2993,4.128410,3017,12.498887,3017,180,5.966192


In [103]:
# Inspect the existing Python payment-method experience summary before SQL–Python reconciliation.
print("payment_method_experience_summary shape: ", payment_method_experience_summary.shape)
print("payment_method_experience_summary payment methods: ", payment_method_experience_summary['dominant_payment_type'])
print("payment_method_experience_summary columns: ", payment_method_experience_summary.columns.to_list())
display(payment_method_experience_summary)

payment_method_experience_summary shape:  (4, 12)
payment_method_experience_summary payment methods:  0         boleto
1    credit_card
2     debit_card
3        voucher
Name: dominant_payment_type, dtype: str
payment_method_experience_summary columns:  ['dominant_payment_type', 'order_count', 'mean_merchandise_value', 'median_merchandise_value', 'median_freight_burden_pct', 'review_eligible_order_count', 'mean_order_review_score', 'delivery_duration_eligible_order_count', 'mean_delivery_duration_days', 'delivery_status_eligible_order_count', 'late_order_count', 'late_delivery_rate_pct']


,dominant_payment_type,order_count,mean_merchandise_value,median_merchandise_value,median_freight_burden_pct,review_eligible_order_count,mean_order_review_score,delivery_duration_eligible_order_count,mean_delivery_duration_days,delivery_status_eligible_order_count,late_order_count,late_delivery_rate_pct
0,boleto,19191,121.401055,74.90,25.282528,19062,4.156070,19191,13.491950,19191,1405.0,7.321140
1,credit_card,72785,143.235029,89.90,21.611111,72298,4.155659,72778,12.349937,72778,4869.0,6.690209
2,debit_card,1484,119.857453,70.45,24.905890,1478,4.241881,1483,10.788641,1483,79.0,5.327040
3,voucher,3017,95.566947,59.99,27.398058,2993,4.128411,3017,12.498887,3017,180.0,5.966192


In [ ]:
# Match the SQL and Python experience summaries by dominant payment method.
payment_method_experience_comparison = sql_payment_method_experience_summary.merge(payment_method_experience_summary, 
                                                                                   on='dominant_payment_type',
                                                                                   how='outer',
                                                                                   validate='one_to_one',
                                                                                   suffixes=('_sql', '_python'),
                                                                                   indicator=True)

print("payment_method_experience_comparison shape: ", payment_method_experience_comparison.shape)
print("payment_method_experience_comparison duplicated payment methods: ", payment_method_experience_comparison['dominant_payment_type'].duplicated().sum())
print("payment_method_experience_comparison merge indicator: ", payment_method_experience_comparison['_merge'].value_counts())

payment_method_experience_comparison shape:  (4, 24)
payment_method_experience_comparison duplicated payment methods:  0
payment_method_experience_comparison merge indicator:  _merge
both          4
left_only     0
right_only    0
Name: count, dtype: int64
